# 🍽️ مُرشِد — نظام ترشيح المطاعم الذكي (Smart Restaurant Recommender)
### مقرر أساسيات LangChain — المشروع النهائي
### جامعة الأمير مساعد بن عبدالرحمن | الكلية التطبيقية | دبلوم الذكاء الاصطناعي التوليدي
**إشراف:** د. محمد الشهري

| العضو | الرقم الجامعي | الجزئية |
|---|---|---|
| همس الشهري | 251001109 | الإعداد وربط النموذج، ملف `.env`، الدمج والتسليم |
| يارا المالكي | 251000183 | البيانات، Document Loader، Embeddings، Chroma، Retriever |
| لمى الأسمري | 251000016 | قاعدة البيانات SQLite، الذاكرة، معالجة الأخطاء، الاختبارات |
| عائشة المالكي | 251004046 | هندسة الأوامر (Prompts)، السلاسل (LCEL Chains)، المخرجات المنظمة |
| رنيم حكمي | 251000215 | تصميم الواجهة في فيقما وتنفيذها في Streamlit |

**معمارية النظام:**
```
المستخدم (Streamlit) ──► التحقق من المدخلات ──► Extraction Chain (Prompt | LLM → UserPreferences)
        ▲                                                   │
        │                                                   ▼
 بطاقات المطاعم ◄── Guardrail ◄── Recommendation Chain ◄── Retriever (Chroma + فلترة metadata)
        │                          (Prompt + Context + Memory)
        ▼
   SQLite: رسائل · ترشيحات · مفضلة · تقييم        ← وعند تعطل OpenAI: البحث المحلي (fallback)
```

**طريقة التشغيل:**
1. أضيفي مفتاح OpenAI في **Colab Secrets** (أيقونة 🔑 يسار الشاشة) باسم `OPENAI_API_KEY` وفعّلي **Notebook access**.
2. من القائمة: **Runtime ← Run all**.
3. في آخر الدفتر يطلع لك زر يفتح واجهة النظام.

## 🎬 الديمو التعريفي المتحرك
عرض بصري سريع للمشروع (الفكرة، المشكلة، رحلة الطلب، المحادثة، قاعدة البيانات، الأرقام). **ما يحتاج مفتاح API.**

In [ ]:
#@title الديمو التعريفي المتحرك (الكود مخفي) { display-mode: "form" }
import html
from pathlib import Path
from IPython.display import HTML, display

DEMO = r'''<!DOCTYPE html>
<html lang="ar" dir="rtl"><head><meta charset="utf-8">
<link href="https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&display=swap" rel="stylesheet">
<style>
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:Tajawal,sans-serif;background:#091418;color:#fff;overflow:hidden}
.stage{position:relative;width:100%;height:640px;overflow:hidden;
 background:radial-gradient(ellipse at 90% -10%,rgba(18,128,138,.55),transparent 55%),radial-gradient(ellipse at 5% 110%,rgba(227,163,58,.18),transparent 50%),#091418}
.scene{position:absolute;inset:0;padding:40px 54px;opacity:0;visibility:hidden;transition:opacity .6s}
.scene.on{opacity:1;visibility:visible}
.brand{position:absolute;top:22px;left:30px;letter-spacing:6px;color:#E3A33A;font-size:12px}
h1{font-size:46px;font-weight:800}
h2{font-size:32px;font-weight:800;margin-bottom:6px}
.sub{color:#9FB3B9;font-size:16px;margin-bottom:26px}
.saf{color:#E3A33A}.teal{color:#2BB3B1}.ok{color:#7CE0B5}
.card{background:rgba(17,36,43,.85);border:1px solid #24404A;border-radius:16px;padding:18px 20px}
.pill{display:inline-block;border:1px solid #E3A33A;color:#E3A33A;background:rgba(227,163,58,.12);border-radius:999px;padding:5px 14px;font-size:13px;font-weight:700;margin:3px}
.pill.t{border-color:#2BB3B1;color:#2BB3B1;background:rgba(43,179,177,.12)}
.in{opacity:0;transform:translateY(14px)}
.on .in{animation:up .6s forwards}
@keyframes up{to{opacity:1;transform:none}}
.d1{animation-delay:.2s!important}.d2{animation-delay:.6s!important}.d3{animation-delay:1s!important}.d4{animation-delay:1.4s!important}.d5{animation-delay:1.8s!important}.d6{animation-delay:2.2s!important}.d7{animation-delay:2.6s!important}.d8{animation-delay:3s!important}
/* scene 1 */
.cover{display:flex;align-items:center;gap:50px;height:100%}
.cover img{width:300px}
/* grids */
.g3{display:grid;grid-template-columns:repeat(3,1fr);gap:16px}
.g6{display:grid;grid-template-columns:repeat(6,1fr);gap:10px}
.g4{display:grid;grid-template-columns:repeat(4,1fr);gap:16px}
.g5{display:grid;grid-template-columns:repeat(5,1fr);gap:12px}
.step{text-align:center;padding:16px 8px;position:relative}
.step .n{position:absolute;top:8px;right:12px;color:#E3A33A;font-weight:800;font-size:13px}
.step .ic{font-size:30px;margin:4px 0 8px}
.step b{display:block;font-size:16px}.step span{display:block;color:#9FB3B9;font-size:12px;margin-top:4px;direction:ltr}
.flowline{height:3px;background:linear-gradient(90deg,#E3A33A,#2BB3B1);border-radius:3px;margin:18px 0;transform-origin:right;transform:scaleX(0)}
.on .flowline{animation:grow 2.6s .2s forwards}
@keyframes grow{to{transform:scaleX(1)}}
/* chat */
.bubble{background:#0F3D4C;border-radius:18px 18px 4px 18px;padding:12px 18px;display:inline-block;font-size:16px;margin-bottom:14px}
.typing::after{content:"|";animation:blink 1s infinite}
@keyframes blink{50%{opacity:0}}
.rc{background:#fff;color:#1D2A32;border-radius:16px;padding:14px 16px}
.rc .top{display:flex;justify-content:space-between;font-size:12px;font-weight:700}
.rc .rk{background:#FCF1DE;color:#9A6412;border-radius:999px;padding:2px 8px}
.rc .m{color:#12808A}
.rc h4{font-size:19px;font-weight:800;margin:8px 0 2px}
.rc .meta{color:#6B7780;font-size:12px}
.bar{height:6px;background:#ECE7DD;border-radius:9px;margin:10px 0;overflow:hidden}
.bar i{display:block;height:6px;width:0;background:linear-gradient(90deg,#12808A,#5B4B9A);border-radius:9px}
.on .bar i{animation:barw 1.2s 2.4s forwards}
@keyframes barw{to{width:var(--w)}}
.rc p{font-size:12.5px;color:#46535C;line-height:1.6}
/* db */
.tb{background:#fff;color:#1D2A32;border-radius:10px;overflow:hidden;font-family:Consolas,monospace;font-size:12px}
.tb .h{background:#0F3D4C;color:#fff;padding:8px;text-align:center;font-weight:700}
.tb div.r{padding:4px 10px;direction:ltr;text-align:left;border-top:1px solid #eee}
/* stats */
.big{font-size:54px;font-weight:800;line-height:1.1}
.nav{position:absolute;bottom:16px;left:0;right:0;display:flex;justify-content:center;gap:8px}
.nav b{width:28px;height:6px;border-radius:6px;background:#24404A;cursor:pointer}
.nav b.on{background:#E3A33A}
.foot{position:absolute;bottom:14px;right:54px;color:#9FB3B9;font-size:11px}
</style></head><body>
<div class="stage">
<div class="brand">M U R S H I D</div>

<section class="scene on">
 <div class="cover">
  <div style="flex:1">
   <div class="in d1 sub" style="margin-bottom:8px">المشروع النهائي | مقرر أساسيات LangChain | الكلية التطبيقية — دبلوم الذكاء الاصطناعي التوليدي</div>
   <h1 class="in d2">نظام ترشيح المطاعم الذكي</h1>
   <div class="in d3" style="font-size:20px;color:#9FB3B9;margin:10px 0 22px">«مُرشِد» يفهم طلبك بالعربي، يبحث في بيانات المطاعم، ويرشّح لك الأنسب مع السبب</div>
   <div class="in d4"><span class="pill">LangChain</span><span class="pill">OpenAI</span><span class="pill">Chroma</span><span class="pill">SQLite</span><span class="pill">Streamlit</span></div>
   <div class="in d5 card" style="margin-top:22px;font-size:14px;color:#9FB3B9">يارا المالكي · همس الشهري · عائشة المالكي · رنيم حكمي · لمى الأسمري &nbsp;|&nbsp; إشراف: د. محمد الشهري</div>
  </div>
  <img class="in d2" src="..." alt="شعار الجامعة">
 </div>
</section>

<section class="scene">
 <h2 class="in d1">المشكلة… والحل</h2><div class="sub in d1">ثلاث صعوبات تواجه أي شخص يدوّر مطعم</div>
 <div class="g3">
  <div class="card in d2"><div style="font-size:30px">🧩</div><h3 style="margin:8px 0">فلاتر جامدة</h3><p style="color:#9FB3B9;font-size:14px">أنت تفكر بجملة: «مكان هادئ وما يكون غالي»، والتطبيق يبي قوائم.</p></div>
  <div class="card in d3"><div style="font-size:30px">❔</div><h3 style="margin:8px 0">بدون تفسير</h3><p style="color:#9FB3B9;font-size:14px">قائمة طويلة، وما تعرف ليش هذا المطعم يناسبك.</p></div>
  <div class="card in d4"><div style="font-size:30px">⚠️</div><h3 style="margin:8px 0">هلوسة النماذج</h3><p style="color:#9FB3B9;font-size:14px">النموذج العام ممكن يخترع مطاعم أو أسعار غير موجودة.</p></div>
 </div>
 <div class="card in d6" style="margin-top:22px;border-color:#E3A33A;background:rgba(227,163,58,.08)">
  <b class="saf">حلّنا ⚡</b><p style="font-size:17px;margin-top:6px">مُرشِد يحوّل الجملة لشروط، يرشّح فقط من بيانات حقيقية (RAG)، يشرح السبب، ويتذكر المحادثة.</p>
 </div>
</section>

<section class="scene">
 <h2 class="in d1">رحلة الطلب داخل النظام</h2><div class="sub in d1">ست خطوات من الكتابة إلى البطاقات</div>
 <div class="flowline"></div>
 <div class="g6">
  <div class="card step in d1"><span class="n">1</span><div class="ic">💬</div><b>الواجهة</b><span>Streamlit</span></div>
  <div class="card step in d2"><span class="n">2</span><div class="ic">🛡️</div><b>التحقق</b><span>validate_user_input</span></div>
  <div class="card step in d3"><span class="n">3</span><div class="ic">🧠</div><b>الاستخراج</b><span>UserPreferences</span></div>
  <div class="card step in d4"><span class="n">4</span><div class="ic">🔎</div><b>البحث</b><span>Chroma + filter</span></div>
  <div class="card step in d5" style="border-color:#E3A33A"><span class="n">5</span><div class="ic">🍽️</div><b>الترشيح</b><span>RecommendationResponse</span></div>
  <div class="card step in d6"><span class="n">6</span><div class="ic">✅</div><b>الحماية والحفظ</b><span>Guardrail + SQLite</span></div>
 </div>
 <div class="card in d7" style="margin-top:22px;direction:ltr;text-align:left;font-family:Consolas,monospace;font-size:13px;background:#050C0F;line-height:1.9">
  <span class="saf">extraction_chain</span> = EXTRACTION_PROMPT | llm.with_structured_output(UserPreferences)<br>
  <span class="saf">recommendation_chain</span> = RECOMMENDATION_PROMPT | llm.with_structured_output(RecommendationResponse)<br>
  <div class="ok" dir="rtl" style="text-align:right;font-family:Tajawal"># إذا تعطل OpenAI ننتقل للبحث المحلي، والنظام ما يوقف</div>
 </div>
</section>

<section class="scene">
 <h2 class="in d1">مثال على المحادثة</h2><div class="sub in d1">هكذا يظهر الرد في الواجهة</div>
 <div class="bubble in d1"><span id="typed"></span></div>
 <div class="in d3" style="margin-bottom:12px"><b style="color:#9FB3B9;font-size:14px">فهمت طلبك: </b><span class="pill t">جدة</span><span class="pill t">مأكولات بحرية</span><span class="pill t">مناسب للعائلة</span><span class="pill t">سعر متوسط</span></div>
 <div class="g3">
  <div class="rc in d4"><div class="top"><span class="rk">الترشيح 1</span><span class="m">تطابق 92%</span></div><h4>لؤلؤة البحر</h4><div class="meta">مأكولات بحرية · جدة - الكورنيش · 4.6/5</div><div class="bar"><i style="--w:92%"></i></div><p>سمك طازج تختاره بنفسك، وجلسات عائلية مطلة على الكورنيش.</p></div>
  <div class="rc in d5"><div class="top"><span class="rk">الترشيح 2</span><span class="m">تطابق 84%</span></div><h4>مذاق الهند</h4><div class="meta">هندي · جدة - الشرفية · 4.4/5</div><div class="bar"><i style="--w:84%"></i></div><p>أرخص ومناسب للعائلات، وفيه أطباق بحرية وكاري.</p></div>
  <div class="rc in d6"><div class="top"><span class="rk">الترشيح 3</span><span class="m">تطابق 71%</span></div><h4>بستان الشام</h4><div class="meta">شامي · جدة - الروضة · 4.3/5</div><div class="bar"><i style="--w:71%"></i></div><p>بديل عائلي بسعر متوسط وحديقة خارجية.</p></div>
 </div>
 <div class="foot" style="right:auto;left:54px">* مثال توضيحي لشكل الواجهة</div>
</section>

<section class="scene">
 <h2 class="in d1">قاعدة البيانات SQLite</h2><div class="sub in d1">خمسة جداول تحفظ كل شي</div>
 <div class="g5">
  <div class="tb in d1"><div class="h">sessions</div><div class="r">id</div><div class="r">created_at</div></div>
  <div class="tb in d2"><div class="h">messages</div><div class="r">session_id</div><div class="r">role</div><div class="r">content</div></div>
  <div class="tb in d3"><div class="h">recommendations</div><div class="r">query</div><div class="r">restaurant_id</div><div class="r">reason</div><div class="r">match_score</div><div class="r">source</div></div>
  <div class="tb in d4"><div class="h">favorites</div><div class="r">session_id</div><div class="r">restaurant_id</div></div>
  <div class="tb in d5"><div class="h">feedback</div><div class="r">query</div><div class="r">is_helpful</div></div>
 </div>
 <div class="g3" style="margin-top:22px">
  <div class="card in d6"><b class="saf">الذاكرة</b><p style="color:#9FB3B9;font-size:14px;margin-top:6px">آخر 6 رسائل من جدول messages تدخل الـ Prompt</p></div>
  <div class="card in d7"><b class="saf">المفضلة</b><p style="color:#9FB3B9;font-size:14px;margin-top:6px">مفتاح مركب يمنع التكرار</p></div>
  <div class="card in d8"><b class="saf">الحماية</b><p style="color:#9FB3B9;font-size:14px;margin-top:6px">commit / rollback وأي خطأ ← رسالة عربية</p></div>
 </div>
</section>

<section class="scene">
 <h2 class="in d1">الجودة بالأرقام</h2><div class="sub in d1">من المشروع نفسه</div>
 <div class="g4">
  <div class="card in d2" style="text-align:center;border-color:#7CE0B5"><div class="big ok"><span class="cnt" data-to="18">0</span>/18</div><div style="color:#9FB3B9">اختبار آلي ناجح</div></div>
  <div class="card in d3" style="text-align:center"><div class="big saf cnt" data-to="8">0</div><div style="color:#9FB3B9">مكونات LangChain</div></div>
  <div class="card in d4" style="text-align:center"><div class="big teal cnt" data-to="5">0</div><div style="color:#9FB3B9">أخطاء متوقعة معالجة</div></div>
  <div class="card in d5" style="text-align:center"><div class="big saf cnt" data-to="36">0</div><div style="color:#9FB3B9">مطعمًا في 9 مدن</div></div>
 </div>
 <div class="card in d6" style="margin-top:24px;text-align:center;font-size:17px">شغّلي الخلايا تحت بالترتيب — كل جزئية مع اختباراتها، وفي الأخير رابط الواجهة 🚀</div>
</section>

<div class="nav"></div>
</div>
<script>
const scenes=[...document.querySelectorAll('.scene')],nav=document.querySelector('.nav');let cur=0,timer;
scenes.forEach((_,i)=>{const b=document.createElement('b');b.onclick=()=>{go(i);restart()};nav.appendChild(b)});
function go(i){scenes[cur].classList.remove('on');cur=i;scenes.forEach(s=>s.classList.remove('on'));void scenes[i].offsetWidth;scenes[i].classList.add('on');
 [...nav.children].forEach((b,k)=>b.classList.toggle('on',k===i));
 if(i===3)typeIt();if(i===5)count();}
function restart(){clearInterval(timer);timer=setInterval(()=>go((cur+1)%scenes.length),7000)}
function typeIt(){const t='أبي مطعم بحري في جدة مناسب للعائلة وسعره متوسط',el=document.getElementById('typed');el.className='typing';el.textContent='';let k=0;
 const iv=setInterval(()=>{el.textContent=t.slice(0,++k);if(k>=t.length){clearInterval(iv);el.className=''}},45)}
function count(){document.querySelectorAll('.cnt').forEach(e=>{const to=+e.dataset.to;let v=0;e.textContent='0';
 setTimeout(()=>{const iv=setInterval(()=>{v++;e.textContent=v;if(v>=to)clearInterval(iv)},Math.max(20,900/to))},800)})}
go(0);restart();
</script>
</body></html>
'''
Path("demo.html").write_text(DEMO, encoding="utf-8")
display(HTML('<iframe srcdoc="' + html.escape(DEMO) + '" width="100%" height="660" '
             'style="border:1px solid #24404A;border-radius:16px"></iframe>'))

## 📦 تثبيت المكتبات

In [ ]:
!pip install -q -U langchain-core langchain-openai langchain-chroma chromadb streamlit pandas pydantic python-dotenv pytest
print("تم تثبيت المكتبات ✓")

---
# 1) جزئية همس الشهري — الإعداد وربط النموذج
1. ملف الإعدادات `config.py` (قراءة المفاتيح من `.env` أو Secrets — بدون أي مفتاح داخل الكود)
2. ملف الأخطاء المشترك `errors.py` (كل الأجزاء تستخدمه)
3. ملفات المشروع: `requirements.txt` و `.env.example` و `.gitignore`

In [ ]:
#@title config.py · errors.py · requirements.txt (الكود مخفي) { display-mode: "form" }
from pathlib import Path

def _w(path, text):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8'); print('✓', path)

_w('src/__init__.py', r'''''')
_w('src/config.py', r'''"""
config.py — إعدادات المشروع (المسؤولة: همس الشهري)

يقرأ الإعدادات من ملف .env محليًا، أو من Secrets في Streamlit Cloud.
لا يوجد أي مفتاح API مكتوب داخل الكود.
"""
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

# مسار جذر المشروع (المجلد الذي فيه app.py)
BASE_DIR = Path(__file__).resolve().parent.parent

# تحميل ملف .env إن وُجد (python-dotenv اختيارية حتى لا يتعطل البرنامج بدونها)
try:
    from dotenv import load_dotenv

    load_dotenv(BASE_DIR / ".env")
except ImportError:  # pragma: no cover
    pass


def _read_secret(name: str, default: str = "") -> str:
    """يقرأ القيمة من متغيرات البيئة أولًا ثم من st.secrets (Streamlit Cloud)."""
    value = os.getenv(name)
    if value:
        return value
    try:
        import streamlit as st

        if name in st.secrets:
            return str(st.secrets[name])
    except Exception:
        # لا يوجد ملف secrets أو streamlit غير مثبتة — نكمل بالقيمة الافتراضية
        pass
    return default


@dataclass(frozen=True)
class Settings:
    """كل إعدادات التطبيق في مكان واحد."""

    openai_api_key: str
    chat_model: str
    embedding_model: str
    temperature: float
    data_path: Path
    chroma_dir: Path
    db_path: Path
    top_k: int
    history_window: int

    @property
    def has_api_key(self) -> bool:
        return bool(self.openai_api_key and self.openai_api_key.startswith("sk-"))


def get_settings() -> Settings:
    """ينشئ كائن الإعدادات. يُستدعى مرة واحدة عند تشغيل التطبيق."""
    return Settings(
        openai_api_key=_read_secret("OPENAI_API_KEY"),
        chat_model=_read_secret("OPENAI_MODEL", "gpt-4o-mini"),
        embedding_model=_read_secret("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
        temperature=float(_read_secret("LLM_TEMPERATURE", "0.3")),
        data_path=BASE_DIR / "data" / "restaurants.csv",
        chroma_dir=BASE_DIR / "chroma_db",
        db_path=BASE_DIR / "restaurant_app.db",
        top_k=int(_read_secret("RETRIEVER_TOP_K", "6")),
        history_window=int(_read_secret("HISTORY_WINDOW", "6")),
    )
''')
_w('src/errors.py', r'''"""
errors.py — الأخطاء المخصصة ورسائلها بالعربي (المسؤولة: لمى الأسمري)

كل خطأ متوقع في النظام له Class خاص ورسالة واضحة للمستخدم،
حتى لا يظهر للمستخدم Traceback إنجليزي غير مفهوم.
"""


class AppError(Exception):
    """الأب لكل أخطاء التطبيق. user_message هي الرسالة التي تظهر في الواجهة."""

    user_message = "حدث خطأ غير متوقع، حاول مرة أخرى."

    def __init__(self, detail: str = ""):
        super().__init__(detail or self.user_message)
        self.detail = detail


class MissingAPIKeyError(AppError):
    user_message = (
        "مفتاح OpenAI غير موجود. أضيفي OPENAI_API_KEY في ملف ‎.env‎ "
        "أو في Secrets على Streamlit Cloud. يعمل النظام الآن بوضع البحث المحلي."
    )


class LLMUnavailableError(AppError):
    user_message = (
        "تعذّر الاتصال بالنموذج اللغوي (OpenAI) حاليًا. "
        "عرضنا لك نتائج من البحث المحلي بدلًا من ذلك."
    )


class InvalidUserInputError(AppError):
    user_message = "الطلب غير واضح. اكتب ما تبحث عنه، مثل: مطعم بحري عائلي في جدة بسعر متوسط."


class UnsupportedFileError(AppError):
    user_message = "نوع الملف غير مدعوم. الملفات المقبولة: CSV فقط وبنفس أعمدة ملف المطاعم."


class DatabaseUnavailableError(AppError):
    user_message = "قاعدة البيانات غير متاحة حاليًا، لن يتم حفظ السجل في هذه الجلسة."


class InvalidModelResponseError(AppError):
    user_message = (
        "رد النموذج لم يكن بالشكل المتوقع، فاستخدمنا نتائج البحث مباشرة. "
        "جرّب إعادة صياغة الطلب."
    )
''')
_w('requirements.txt', r'''# واجهة المستخدم
streamlit>=1.40

# LangChain ومكوناته
langchain-core>=0.3
langchain-openai>=0.3
langchain-chroma>=0.2
chromadb>=0.5

# البيانات والإعدادات
pandas>=2.0
pydantic>=2.0
python-dotenv>=1.0

# Streamlit Cloud: نسخة sqlite حديثة يحتاجها Chroma (على Linux فقط)
pysqlite3-binary; platform_system == "Linux"

# الاختبارات
pytest>=8.0
''')
_w('.env.example', r'''# انسخي هذا الملف إلى .env وضعي مفتاحك الحقيقي. لا ترفعي .env على GitHub.
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o-mini
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
LLM_TEMPERATURE=0.3
RETRIEVER_TOP_K=6
HISTORY_WINDOW=6
''')
_w('.gitignore', r'''.env
.streamlit/secrets.toml
__pycache__/
*.pyc
.pytest_cache/
chroma_db/
chroma_db_uploaded/
*.db
data/uploaded_restaurants.csv
.venv/
venv/
''')
_w('.streamlit/config.toml', r'''[server]
enableStaticServing = true
headless = true
port = 8501
enableCORS = false
enableXsrfProtection = false

[browser]
gatherUsageStats = false

[theme]
base = "light"
primaryColor = "#12808A"
backgroundColor = "#F7F4EE"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#1D2A32"
font = "sans serif"
''')

### 1.1 قراءة المفتاح من Colab Secrets وحفظه في `.env`
(الطريقة الآمنة المطلوبة في الإرشادات — لا يوجد مفتاح داخل الكود)

In [ ]:
import os
from pathlib import Path
from getpass import getpass

key = ""
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY") or ""
except Exception:
    pass
if not key:
    key = getpass("ما لقيت المفتاح في Secrets — الصقيه هنا (يبدأ بـ sk-) أو اضغطي Enter للتجربة بدونه: ")

key = key.strip()
Path(".env").write_text(f"OPENAI_API_KEY={key}\nOPENAI_MODEL=gpt-4o-mini\n", encoding="utf-8")
os.environ["OPENAI_API_KEY"] = key
print("المفتاح جاهز ✓" if key.startswith("sk-") else "⚠️ بدون مفتاح: النظام بيشتغل بوضع البحث المحلي فقط")

### 1.2 اختبار ربط النموذج

In [ ]:
import importlib, src.config as config
importlib.reload(config)
settings = config.get_settings()
print("النموذج:", settings.chat_model, "| التضمين:", settings.embedding_model)

if settings.has_api_key:
    from langchain_openai import ChatOpenAI
    try:
        llm = ChatOpenAI(model=settings.chat_model, api_key=settings.openai_api_key, timeout=30)
        print("رد النموذج:", llm.invoke("رد بكلمة واحدة فقط: جاهز").content, "✓")
    except Exception as error:
        print("ما اشتغل الاتصال:", error)
else:
    print("ما فيه مفتاح — تخطّينا اختبار الاتصال")

---
# 2) جزئية يارا المالكي — البيانات والبحث الدلالي
1. ملف بيانات المطاعم `restaurants.csv` (36 مطعمًا — بيانات تجريبية بأسماء افتراضية أعدّها الفريق)
2. Document Loader: تحويل كل مطعم إلى `Document`
3. Embeddings + Chroma + Retriever مع فلترة metadata وتخفيف تدريجي

In [ ]:
#@title data/restaurants.csv (البيانات مخفية) { display-mode: "form" }
from pathlib import Path

def _w(path, text):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8'); print('✓', path)

_w('data/restaurants.csv', r'''id,name,cuisine,city,district,price_level,avg_price_sar,rating,vegetarian_options,family_friendly,delivery,outdoor_seating,opening_hours,specialties,description
1,بيت المندي الأصيل,سعودي,الرياض,العليا,2,55,4.5,لا,نعم,نعم,لا,12:00-01:00,مندي لحم؛ مندي دجاج؛ مظبي,مطعم شعبي يقدم المندي المطبوخ في التنور على الطريقة الحضرمية مع جلسات أرضية عائلية وأقسام خاصة.
2,نكهة الحجاز,سعودي,جدة,البلد,2,45,4.6,نعم,نعم,لا,نعم,08:00-00:00,فول بالسمن؛ مطبق؛ معصوب؛ سوبيا,مطعم تراثي في قلب جدة التاريخية يقدم الأكلات الحجازية والفطور الشعبي في أجواء البيوت القديمة.
3,سفرة مكة,سعودي,مكة المكرمة,العزيزية,2,50,4.4,نعم,نعم,نعم,لا,11:00-02:00,سليق؛ رز بخاري؛ كبسة لحم,مطعم عائلي قريب من الطرق المؤدية للحرم يقدم أطباق مكية تقليدية بكميات مناسبة للعائلات والمجموعات.
4,ساكورا هاوس,ياباني,الرياض,حطين,4,180,4.7,نعم,لا,لا,نعم,13:00-00:00,سوشي؛ رامن؛ تيمبورا؛ واغيو,مطعم ياباني راقٍ بإطلالة خارجية يقدم السوشي الطازج وأطباق الواغيو مع ركن للطبخ المباشر أمام الضيوف.
5,بستان الشام,شامي,جدة,الروضة,2,60,4.3,نعم,نعم,نعم,نعم,12:00-01:00,مشاوي مشكلة؛ فتوش؛ حمص؛ كبة,مطعم شامي بحديقة خارجية يقدم المشاوي والمقبلات الباردة والحارة مع خيارات نباتية كثيرة.
6,بيتزا نابولي,إيطالي,الرياض,الملقا,3,90,4.5,نعم,نعم,نعم,لا,12:00-00:00,بيتزا مارغريتا؛ باستا كاربونارا؛ تيراميسو,مطعم إيطالي يخبز البيتزا في فرن حطب نابولي ويقدم الباستا الطازجة المصنوعة يوميًا.
7,برجر ستيشن,أمريكي,الدمام,الشاطئ,2,48,4.2,نعم,نعم,نعم,نعم,13:00-03:00,برجر لحم مدخن؛ بطاطس بالجبن؛ ميلك شيك,مطعم برجر عصري قريب من الكورنيش يناسب الشباب ويقدم خيار برجر نباتي وجلسات خارجية مسائية.
8,مذاق الهند,هندي,جدة,الشرفية,2,52,4.4,نعم,نعم,نعم,لا,12:00-00:30,برياني دجاج؛ بتر تشكن؛ نان بالثوم؛ دال,مطعم هندي يقدم البرياني والكاري بدرجات حرارة مختلفة مع قائمة نباتية واسعة.
9,لؤلؤة البحر,مأكولات بحرية,جدة,الكورنيش,3,120,4.6,لا,نعم,لا,نعم,13:00-01:00,سمك حريد مشوي؛ روبيان جامبو؛ صيادية,مطعم مأكولات بحرية على الواجهة البحرية يختار فيه الزبون السمك الطازج ويحدد طريقة الطهي.
10,كافيه ورق,مقهى,الرياض,النخيل,2,40,4.3,نعم,نعم,نعم,نعم,07:00-00:00,قهوة مختصة؛ كرواسون؛ سلطة كينوا؛ ساندويتش,مقهى هادئ مناسب للمذاكرة والعمل يقدم قهوة مختصة وفطورًا خفيفًا وخيارات صحية.
11,دار الكبسة,سعودي,الدمام,الفيصلية,1,35,4.1,لا,نعم,نعم,لا,11:00-00:00,كبسة دجاج؛ مثلوثة؛ جريش,مطعم اقتصادي يقدم الكبسة والأكلات النجدية اليومية بأسعار مناسبة للطلاب والعائلات.
12,تاكو لوكو,مكسيكي,الرياض,الياسمين,2,65,4.2,نعم,لا,نعم,نعم,14:00-02:00,تاكو؛ بوريتو؛ ناتشوز؛ كيساديا,مطعم مكسيكي بأجواء شبابية يقدم أطباقًا حارة وخيارات نباتية من الفاصوليا والخضار المشوية.
13,قصر الشاورما,شامي,مكة المكرمة,الشوقية,1,22,4.0,نعم,نعم,نعم,لا,10:00-03:00,شاورما دجاج؛ شاورما لحم؛ فلافل,مطعم سريع واقتصادي يقدم الشاورما والفلافل ومناسب للوجبات السريعة بعد صلاة العشاء.
14,حديقة التوابل,تايلندي,جدة,الزهراء,3,95,4.5,نعم,نعم,لا,نعم,13:00-00:00,باد تاي؛ كاري أخضر؛ توم يم؛ أرز مانجو,مطعم تايلندي بديكور استوائي يقدم أطباقًا متوازنة بين الحار والحامض مع قائمة نباتية مخصصة.
15,ستيك هاوس النخبة,ستيك,الرياض,الورود,4,220,4.8,لا,لا,لا,لا,18:00-01:00,ريب آي؛ تي بون؛ ستيك واغيو,مطعم ستيك فاخر مناسب للمناسبات الخاصة واجتماعات العمل مع لحوم معتّقة داخليًا.
16,فطور البيت,سعودي,أبها,المنسك,1,30,4.4,نعم,نعم,نعم,نعم,06:00-13:00,عريكة؛ معصوب؛ شاي جبلي؛ خبز تنور,مطعم فطور جنوبي بإطلالة جبلية يقدم العريكة بالعسل والسمن البلدي وخبز التنور الساخن.
17,ميزان التركي,تركي,الرياض,السليمانية,2,70,4.4,نعم,نعم,نعم,لا,12:00-01:00,إسكندر كباب؛ بيدا؛ عدنة؛ كنافة,مطعم تركي يقدم الكباب والبيدا المخبوزة في الفرن الحجري مع حلويات تركية طازجة.
18,سلطة ومزاج,صحي,جدة,السلامة,2,55,4.3,نعم,نعم,نعم,لا,10:00-23:00,بولات صحية؛ سلطات؛ عصائر طبيعية؛ وجبات قليلة السعرات,مطعم صحي يعرض السعرات الحرارية لكل طبق ويقدم خيارات نباتية وخالية من الجلوتين.
19,دمبلنغ هاوس,صيني,الدمام,الخليج,2,58,4.2,نعم,نعم,نعم,لا,12:00-00:00,دمبلنغ؛ نودلز مقلية؛ دجاج كونغ باو,مطعم صيني يقدم الدمبلنغ المصنوع يدويًا والنودلز المقلية بخيارات لحم وخضار.
20,مشويات الطائف,سعودي,الطائف,الشهداء,2,60,4.5,لا,نعم,لا,نعم,16:00-02:00,مشويات لحم حري؛ كبدة؛ كفتة,مطعم مشويات بجلسات خارجية في أجواء الطائف المعتدلة يقدم لحم الحري المحلي.
21,حلويات المدينة,حلويات,المدينة المنورة,قباء,1,25,4.6,نعم,نعم,نعم,لا,09:00-00:00,كنافة؛ بسبوسة؛ معمول تمر؛ قهوة عربية,محل حلويات شرقية يقدم الكنافة الساخنة وحلويات التمر المحلية مع القهوة العربية.
22,باريس بيسترو,فرنسي,الرياض,الصحافة,4,160,4.5,نعم,لا,لا,نعم,12:00-00:00,كروك مسيو؛ ستيك فريت؛ كريم بروليه,مطعم فرنسي بأجواء البيسترو الباريسي مناسب للغداء الهادئ والمناسبات الصغيرة.
23,مطبخ اليمن,يمني,مكة المكرمة,النسيم,1,32,4.3,لا,نعم,نعم,لا,11:00-01:00,حنيذ؛ سلتة؛ فحسة؛ شاي عدني,مطعم يمني شعبي يقدم الحنيذ والسلتة والفحسة الحارة مع شاي عدني بالحليب.
24,سوشي إكسبرس,ياباني,جدة,النهضة,2,75,4.1,نعم,لا,نعم,لا,13:00-00:00,سوشي رولز؛ بوكي بول؛ إيدامامي,مطعم ياباني سريع وسعره متوسط يناسب الطلبات الخارجية والوجبات الفردية.
25,بيت الفلافل,مصري,الرياض,البطحاء,1,18,4.2,نعم,نعم,نعم,لا,06:00-15:00,طعمية؛ فول؛ كشري؛ بابا غنوج,مطعم مصري اقتصادي جدًا للفطور والغداء يقدم الكشري والطعمية بخيارات نباتية بالكامل.
26,أمواج الخبر,مأكولات بحرية,الخبر,الكورنيش,3,110,4.4,لا,نعم,لا,نعم,13:00-01:00,سمك هامور؛ كاليماري؛ مطبق سمك,مطعم بحري بإطلالة على الخليج العربي يقدم الهامور المحلي بطرق طبخ خليجية.
27,زعفران,إيراني,الرياض,العقيق,3,85,4.5,نعم,نعم,نعم,لا,12:00-00:00,جلو كباب؛ أرز بالزعفران؛ قورمه سبزي,مطعم يقدم المطبخ الفارسي بالأرز المعطر بالزعفران والكباب المشوي على الفحم.
28,كوري غريل,كوري,جدة,المحمدية,3,100,4.6,نعم,لا,لا,لا,14:00-01:00,شواء كوري على الطاولة؛ بيبيمباب؛ كيمتشي,مطعم كوري يشوي فيه الزبائن اللحوم على الطاولة مع أطباق جانبية متعددة.
29,مطعم الواحة,سعودي,الأحساء,الهفوف,1,38,4.3,لا,نعم,نعم,نعم,11:00-00:00,مكبوس؛ هريس؛ خبز أحمر؛ رز حساوي,مطعم أحسائي يقدم الأكلات الشرقية التقليدية مثل الهريس والرز الحساوي الأحمر.
30,غرين باول,صحي,الرياض,الربيع,2,50,4.4,نعم,نعم,نعم,لا,09:00-23:00,بولات نباتية؛ سموذي؛ وجبات بروتين,مطعم صحي يناسب الرياضيين والنباتيين ويقدم وجبات محسوبة البروتين والسعرات.
31,ليالي لبنان,لبناني,الدمام,الشاطئ الغربي,3,95,4.5,نعم,نعم,نعم,نعم,12:00-01:00,مشاوي لبنانية؛ مناقيش؛ تبولة؛ حلاوة الجبن,مطعم لبناني بجلسات خارجية وعروض عائلية في نهاية الأسبوع ومقبلات نباتية متنوعة.
32,كرك وفطاير,مقهى,مكة المكرمة,الزاهر,1,15,4.1,نعم,نعم,نعم,لا,15:00-04:00,شاي كرك؛ فطاير جبن؛ مطبق حلو,كشك شعبي يقدم الكرك والفطاير حتى ساعات الفجر ومناسب للوجبات الخفيفة السريعة.
33,ذا فارم كيتشن,عالمي,أبها,السودة,3,105,4.7,نعم,نعم,لا,نعم,08:00-23:00,فطور مزرعة؛ ستيك دجاج؛ بيتزا جبلية,مطعم ريفي في منطقة السودة يستخدم منتجات مزارع محلية مع إطلالة على الجبال والضباب.
34,بخاري الملكي,أفغاني,جدة,الصفا,2,48,4.4,لا,نعم,نعم,لا,11:00-01:00,رز بخاري؛ لحم مقلقل؛ مانتو,مطعم أفغاني يقدم الرز البخاري بالجزر والزبيب واللحم المطبوخ ببطء.
35,بانكيك فاكتوري,حلويات,الرياض,النرجس,2,45,4.3,نعم,نعم,نعم,لا,08:00-00:00,بانكيك؛ وافل؛ كريب؛ آيس كريم,محل حلويات عصري متخصص في البانكيك والوافل ومناسب للأطفال والجلسات الصباحية.
36,مندي ومظبي القرية,سعودي,المدينة المنورة,العزيزية,1,40,4.2,لا,نعم,نعم,لا,11:00-01:00,مندي؛ مظبي؛ مضغوط,مطعم شعبي اقتصادي بأقسام عائلية يقدم المندي والمظبي المطبوخ على الحجر.
''')

In [ ]:
#@title data_loader.py · vector_store.py (الكود مخفي) { display-mode: "form" }
from pathlib import Path

def _w(path, text):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8'); print('✓', path)

_w('src/data_loader.py', r'''"""
data_loader.py — تحميل بيانات المطاعم وتحويلها إلى Documents (المسؤولة: يارا المالكي)

الخطوات:
1) قراءة ملف CSV والتحقق من الأعمدة والقيم.
2) تحويل كل مطعم إلى Document في LangChain:
   - page_content: نص وصفي يُحوَّل لاحقًا إلى Embedding.
   - metadata: الحقول المنظمة (المدينة، السعر، التقييم...) لاستخدامها في الفلترة.
"""
from __future__ import annotations

from pathlib import Path
from typing import List

import pandas as pd

from .errors import UnsupportedFileError

REQUIRED_COLUMNS = [
    "id", "name", "cuisine", "city", "district", "price_level", "avg_price_sar",
    "rating", "vegetarian_options", "family_friendly", "delivery", "outdoor_seating",
    "opening_hours", "specialties", "description",
]

PRICE_LABELS = {1: "اقتصادي", 2: "متوسط", 3: "مرتفع", 4: "فاخر"}


def load_restaurants(path: Path | str) -> pd.DataFrame:
    """يقرأ ملف المطاعم ويتحقق منه. يرمي UnsupportedFileError إذا كان الملف غير صالح."""
    path = Path(path)
    if path.suffix.lower() != ".csv":
        raise UnsupportedFileError(f"امتداد غير مدعوم: {path.suffix}")
    if not path.exists():
        raise UnsupportedFileError(f"الملف غير موجود: {path}")

    try:
        df = pd.read_csv(path, encoding="utf-8")
    except Exception as exc:  # ملف تالف أو ترميز خاطئ
        raise UnsupportedFileError(f"تعذر قراءة الملف: {exc}") from exc

    return validate_dataframe(df)


def validate_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """يتأكد من وجود الأعمدة المطلوبة وينظف القيم."""
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise UnsupportedFileError(f"أعمدة ناقصة: {', '.join(missing)}")

    df = df.copy()
    df = df.dropna(subset=["name", "city", "cuisine"])
    df["price_level"] = pd.to_numeric(df["price_level"], errors="coerce").fillna(2).astype(int).clip(1, 4)
    df["avg_price_sar"] = pd.to_numeric(df["avg_price_sar"], errors="coerce").fillna(0).astype(int)
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce").fillna(0).clip(0, 5)
    for col in ["vegetarian_options", "family_friendly", "delivery", "outdoor_seating"]:
        df[col] = df[col].astype(str).str.strip().isin(["نعم", "yes", "Yes", "1", "True", "true"])
    for col in ["name", "cuisine", "city", "district", "specialties", "description", "opening_hours"]:
        df[col] = df[col].fillna("").astype(str).str.strip()

    if df.empty:
        raise UnsupportedFileError("الملف لا يحتوي على أي مطعم صالح.")
    return df.reset_index(drop=True)


def _yes_no(value: bool) -> str:
    return "نعم" if value else "لا"


def row_to_text(row: pd.Series) -> str:
    """يبني النص الوصفي للمطعم — هذا النص هو ما يُحوَّل إلى Embedding."""
    return (
        f"اسم المطعم: {row['name']}\n"
        f"نوع المطبخ: {row['cuisine']}\n"
        f"الموقع: {row['city']} - حي {row['district']}\n"
        f"مستوى السعر: {PRICE_LABELS.get(int(row['price_level']), 'متوسط')} "
        f"(متوسط {row['avg_price_sar']} ريال للشخص)\n"
        f"التقييم: {row['rating']} من 5\n"
        f"خيارات نباتية: {_yes_no(row['vegetarian_options'])} | "
        f"مناسب للعائلات: {_yes_no(row['family_friendly'])} | "
        f"توصيل: {_yes_no(row['delivery'])} | جلسات خارجية: {_yes_no(row['outdoor_seating'])}\n"
        f"أوقات العمل: {row['opening_hours']}\n"
        f"أشهر الأطباق: {row['specialties'].replace('؛', '،')}\n"
        f"الوصف: {row['description']}"
    )


def row_to_metadata(row: pd.Series) -> dict:
    """البيانات المنظمة المرفقة مع كل Document (تُستخدم للفلترة في Chroma)."""
    return {
        "id": int(row["id"]),
        "name": row["name"],
        "cuisine": row["cuisine"],
        "city": row["city"],
        "district": row["district"],
        "price_level": int(row["price_level"]),
        "avg_price_sar": int(row["avg_price_sar"]),
        "rating": float(row["rating"]),
        "vegetarian_options": bool(row["vegetarian_options"]),
        "family_friendly": bool(row["family_friendly"]),
        "delivery": bool(row["delivery"]),
        "outdoor_seating": bool(row["outdoor_seating"]),
        "opening_hours": row["opening_hours"],
        "specialties": row["specialties"],
    }


def dataframe_to_documents(df: pd.DataFrame) -> List["Document"]:
    """Document Loader: يحوّل جدول المطاعم إلى قائمة Documents في LangChain."""
    from langchain_core.documents import Document  # استيراد متأخر لتسهيل الاختبارات

    return [
        Document(page_content=row_to_text(row), metadata=row_to_metadata(row))
        for _, row in df.iterrows()
    ]
''')
_w('src/vector_store.py', r'''"""
vector_store.py — Embeddings + Chroma + Retriever (المسؤولة: يارا المالكي)

- Embeddings: نحول وصف كل مطعم إلى متجه رقمي باستخدام OpenAI Embeddings.
- Vector Store: نخزن المتجهات في Chroma (قاعدة متجهات محلية تحفظ على القرص).
- Retriever: نبحث عن أقرب المطاعم دلاليًا لطلب المستخدم مع فلترة بالـ metadata.
"""
from __future__ import annotations

import hashlib
import shutil
from pathlib import Path
from typing import List, Optional

from .data_loader import dataframe_to_documents
from .schemas import UserPreferences

COLLECTION_NAME = "restaurants"


def _data_fingerprint(data_path: Path, embedding_model: str) -> str:
    """بصمة للملف والنموذج: إذا تغيّر ملف البيانات نعيد بناء قاعدة المتجهات."""
    digest = hashlib.md5(data_path.read_bytes()).hexdigest()
    return f"{digest}-{embedding_model}"


def build_or_load_vector_store(df, settings):
    """يبني قاعدة Chroma أول مرة، ويعيد استخدامها في المرات التالية."""
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings(model=settings.embedding_model, api_key=settings.openai_api_key)
    persist_dir: Path = settings.chroma_dir
    marker = persist_dir / "fingerprint.txt"
    fingerprint = _data_fingerprint(settings.data_path, settings.embedding_model)

    if marker.exists() and marker.read_text(encoding="utf-8") == fingerprint:
        return Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(persist_dir),
        )

    # البيانات تغيرت أو أول تشغيل: نحذف القديم ونبني من جديد
    if persist_dir.exists():
        shutil.rmtree(persist_dir)
    persist_dir.mkdir(parents=True, exist_ok=True)

    documents = dataframe_to_documents(df)
    store = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(persist_dir),
        ids=[str(d.metadata["id"]) for d in documents],
    )
    marker.write_text(fingerprint, encoding="utf-8")
    return store


def preferences_to_filter(prefs: UserPreferences, level: int = 0) -> Optional[dict]:
    """
    يحوّل التفضيلات إلى فلتر Chroma.
    level يحدد درجة التخفيف عند قلة النتائج:
      0 = كل الشروط، 1 = المدينة + المطبخ + السعر، 2 = المدينة فقط، 3 = بدون فلتر.
    """
    conditions: List[dict] = []
    if prefs.city and level <= 2:
        conditions.append({"city": {"$eq": prefs.city}})
    if level <= 1:
        if prefs.cuisine:
            conditions.append({"cuisine": {"$eq": prefs.cuisine}})
        if prefs.max_price_level:
            conditions.append({"price_level": {"$lte": prefs.max_price_level}})
    if level == 0:
        if prefs.max_budget_sar:
            conditions.append({"avg_price_sar": {"$lte": prefs.max_budget_sar}})
        for field in ("family_friendly", "delivery", "outdoor_seating"):
            if getattr(prefs, field):
                conditions.append({field: {"$eq": True}})
        if prefs.vegetarian:
            conditions.append({"vegetarian_options": {"$eq": True}})

    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    return {"$and": conditions}


def retrieve_restaurants(store, prefs: UserPreferences, k: int = 6, min_results: int = 2):
    """
    Retrieval مع تخفيف تدريجي للفلاتر:
    نبدأ بكل الشروط، وإذا كانت النتائج أقل من min_results نخفف الشروط خطوة خطوة.
    يرجع (المستندات, مستوى التخفيف المستخدم).
    """
    docs = []
    for level in range(4):
        search_kwargs = {"k": k}
        flt = preferences_to_filter(prefs, level)
        if flt:
            search_kwargs["filter"] = flt
        retriever = store.as_retriever(search_type="similarity", search_kwargs=search_kwargs)
        docs = retriever.invoke(prefs.search_query)
        if len(docs) >= min_results:
            return docs, level
    return docs, 3


def format_docs(docs) -> str:
    """يجهز المطاعم المسترجعة كنص للـ Prompt، مع رقم id لكل مطعم."""
    blocks = []
    for d in docs:
        blocks.append(f"[restaurant_id={d.metadata['id']}]\n{d.page_content}")
    return "\n\n---\n\n".join(blocks) if blocks else "لا توجد نتائج."
''')
_w('src/schemas.py', r'''"""
schemas.py — شكل المخرجات المنظمة Structured Output (المسؤولة: عائشة المالكي)

نستخدم Pydantic لتعريف الشكل الذي يجب أن يرجعه النموذج اللغوي،
بدل نص حر يصعب التعامل معه في الواجهة.
"""
from __future__ import annotations

from typing import List, Optional

from pydantic import BaseModel, Field


class UserPreferences(BaseModel):
    """تفضيلات المستخدم المستخرجة من رسالته (المرحلة الأولى في الـ Chain)."""

    city: Optional[str] = Field(None, description="المدينة المطلوبة بالعربي كما في البيانات، مثل: جدة، الرياض، مكة المكرمة")
    cuisine: Optional[str] = Field(None, description="نوع المطبخ بالعربي، مثل: سعودي، ياباني، إيطالي، مأكولات بحرية، صحي")
    max_price_level: Optional[int] = Field(None, ge=1, le=4, description="أعلى مستوى سعر مقبول: 1 اقتصادي، 2 متوسط، 3 مرتفع، 4 فاخر")
    max_budget_sar: Optional[int] = Field(None, ge=0, description="الميزانية للشخص بالريال إن ذكرها المستخدم")
    family_friendly: Optional[bool] = Field(None, description="True إذا طلب مكانًا مناسبًا للعائلة")
    vegetarian: Optional[bool] = Field(None, description="True إذا طلب خيارات نباتية")
    delivery: Optional[bool] = Field(None, description="True إذا طلب توصيل")
    outdoor_seating: Optional[bool] = Field(None, description="True إذا طلب جلسات خارجية")
    occasion: Optional[str] = Field(None, description="المناسبة إن وُجدت: عشاء عمل، عائلة، أصدقاء، فطور، مذاكرة...")
    search_query: str = Field(..., description="جملة بحث قصيرة تلخص ما يريده المستخدم لاستخدامها في البحث الدلالي")
    is_food_related: bool = Field(True, description="False إذا كان الطلب لا علاقة له بالمطاعم أو الأكل")


class RestaurantRecommendation(BaseModel):
    """مطعم واحد مرشّح مع سبب الترشيح."""

    restaurant_id: int = Field(..., description="رقم id للمطعم كما ورد في السياق")
    name: str = Field(..., description="اسم المطعم كما ورد في السياق حرفيًا")
    reason: str = Field(..., description="سبب الترشيح في جملة أو جملتين مرتبطًا بطلب المستخدم")
    suggested_dishes: List[str] = Field(default_factory=list, description="طبق أو طبقان من أشهر أطباق المطعم")
    match_score: int = Field(..., ge=0, le=100, description="مدى تطابق المطعم مع الطلب من 0 إلى 100")


class RecommendationResponse(BaseModel):
    """الرد النهائي الذي يُعرض في الواجهة."""

    summary: str = Field(..., description="رد ودي مختصر للمستخدم بالعربي (جملتان إلى ثلاث)")
    recommendations: List[RestaurantRecommendation] = Field(default_factory=list, max_length=3)
    follow_up_question: Optional[str] = Field(None, description="سؤال متابعة اختياري يساعد على تحسين الترشيح")
''')

### 2.1 تحميل البيانات والتحقق منها

In [ ]:
from src.data_loader import load_restaurants
df = load_restaurants("data/restaurants.csv")
print("عدد المطاعم:", len(df), "| عدد المدن:", df.city.nunique(), "| أنواع المطابخ:", df.cuisine.nunique())
print("\nالمطاعم حسب المدينة:")
print(df.city.value_counts().to_string())
df[["name", "cuisine", "city", "price_level", "avg_price_sar", "rating"]].head(8)

### 2.2 Document Loader — شكل المطعم بعد التحويل

In [ ]:
from src.data_loader import dataframe_to_documents
docs = dataframe_to_documents(df)
print("عدد الـ Documents:", len(docs))
print("\n--- page_content (هذا النص يتحول إلى Embedding) ---")
print(docs[8].page_content)
print("\n--- metadata (تستخدم للفلترة) ---")
print(docs[8].metadata)

### 2.3 بناء الفلتر من التفضيلات (4 مستويات تخفيف)

In [ ]:
from src.schemas import UserPreferences
from src.vector_store import preferences_to_filter

prefs = UserPreferences(city="جدة", cuisine="مأكولات بحرية", max_price_level=3,
                        family_friendly=True, search_query="مطعم بحري عائلي في جدة")
for level in range(4):
    print(f"المستوى {level}:", preferences_to_filter(prefs, level))

### 2.4 بناء Chroma والبحث الدلالي (يحتاج مفتاح)

In [ ]:
from src.config import get_settings
from src.vector_store import build_or_load_vector_store, retrieve_restaurants

settings = get_settings()
if settings.has_api_key:
    store = build_or_load_vector_store(df, settings)
    print("تم بناء قاعدة المتجهات ✓ — عدد المطاعم:", store._collection.count())
    for p in [prefs, UserPreferences(city="الرياض", search_query="مكان هادئ للقهوة والمذاكرة"),
              UserPreferences(city="أبها", cuisine="ياباني", search_query="سوشي")]:
        found, level = retrieve_restaurants(store, p, k=3)
        print(f"\n🔎 {p.search_query} | مستوى التخفيف: {level}")
        for d in found:
            print("  -", d.metadata["name"], "|", d.metadata["cuisine"], "|", d.metadata["city"])
else:
    print("ما فيه مفتاح — تخطّينا بناء Chroma")

### 2.5 اختبار الملف غير المدعوم

In [ ]:
import pandas as pd
from pathlib import Path
from src.errors import UnsupportedFileError
from src.data_loader import validate_dataframe

Path("bad.txt").write_text("hello", encoding="utf-8")
for test in [lambda: load_restaurants("bad.txt"), lambda: load_restaurants("missing.csv"),
             lambda: validate_dataframe(pd.DataFrame({"name": ["x"]}))]:
    try:
        test(); print("FAIL | ما انرفض")
    except UnsupportedFileError as error:
        print("PASS |", error.user_message.split(".")[0], "|", error.detail)

---
# 3) جزئية لمى الأسمري — قاعدة البيانات والذاكرة ومعالجة الأخطاء
1. SQLite بخمسة جداول: sessions · messages · recommendations · favorites · feedback
2. ذاكرة المحادثة `SQLiteChatHistory` (ترث من `BaseChatMessageHistory`)
3. البحث المحلي الاحتياطي عند تعطل OpenAI

In [ ]:
#@title database.py · memory.py · fallback.py (الكود مخفي) { display-mode: "form" }
from pathlib import Path

def _w(path, text):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8'); print('✓', path)

_w('src/database.py', r'''"""
database.py — قاعدة بيانات SQLite (المسؤولة: لمى الأسمري)

تحفظ:
- sessions: جلسات المستخدمين.
- messages: سجل المحادثة (يُستخدم كـ Memory للنموذج).
- recommendations: كل مطعم تم ترشيحه وسببه.
- favorites: المطاعم المفضلة للمستخدم.
- feedback: تقييم المستخدم للترشيحات (👍 / 👎).
"""
from __future__ import annotations

import sqlite3
import uuid
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

from .errors import DatabaseUnavailableError

SCHEMA = """
CREATE TABLE IF NOT EXISTS sessions (
    id          TEXT PRIMARY KEY,
    created_at  TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS messages (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id  TEXT NOT NULL REFERENCES sessions(id),
    role        TEXT NOT NULL CHECK (role IN ('user', 'assistant')),
    content     TEXT NOT NULL,
    created_at  TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS recommendations (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id      TEXT NOT NULL REFERENCES sessions(id),
    query           TEXT NOT NULL,
    restaurant_id   INTEGER NOT NULL,
    restaurant_name TEXT NOT NULL,
    reason          TEXT,
    match_score     INTEGER,
    source          TEXT NOT NULL DEFAULT 'llm',
    created_at      TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS favorites (
    session_id      TEXT NOT NULL REFERENCES sessions(id),
    restaurant_id   INTEGER NOT NULL,
    restaurant_name TEXT NOT NULL,
    created_at      TEXT NOT NULL,
    PRIMARY KEY (session_id, restaurant_id)
);
CREATE TABLE IF NOT EXISTS feedback (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id  TEXT NOT NULL REFERENCES sessions(id),
    query       TEXT NOT NULL,
    is_helpful  INTEGER NOT NULL CHECK (is_helpful IN (0, 1)),
    created_at  TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS idx_messages_session ON messages(session_id, id);
"""


def _now() -> str:
    return datetime.now().isoformat(timespec="seconds")


class Database:
    """غلاف بسيط فوق sqlite3. كل دالة تفتح اتصالًا قصيرًا وتغلقه."""

    def __init__(self, db_path: Path | str):
        self.db_path = str(db_path)
        self.init_schema()

    @contextmanager
    def _connect(self):
        try:
            conn = sqlite3.connect(self.db_path, timeout=10)
            conn.row_factory = sqlite3.Row
            conn.execute("PRAGMA foreign_keys = ON")
        except sqlite3.Error as exc:
            raise DatabaseUnavailableError(str(exc)) from exc
        try:
            yield conn
            conn.commit()
        except sqlite3.Error as exc:
            conn.rollback()
            raise DatabaseUnavailableError(str(exc)) from exc
        finally:
            conn.close()

    def init_schema(self) -> None:
        with self._connect() as conn:
            conn.executescript(SCHEMA)

    # ------------------------------------------------------------- الجلسات
    def create_session(self) -> str:
        session_id = uuid.uuid4().hex
        with self._connect() as conn:
            conn.execute("INSERT INTO sessions (id, created_at) VALUES (?, ?)", (session_id, _now()))
        return session_id

    def ensure_session(self, session_id: str) -> None:
        with self._connect() as conn:
            conn.execute(
                "INSERT OR IGNORE INTO sessions (id, created_at) VALUES (?, ?)", (session_id, _now())
            )

    # ------------------------------------------------------------- الرسائل (Memory)
    def add_message(self, session_id: str, role: str, content: str) -> int:
        with self._connect() as conn:
            cur = conn.execute(
                "INSERT INTO messages (session_id, role, content, created_at) VALUES (?, ?, ?, ?)",
                (session_id, role, content, _now()),
            )
            return int(cur.lastrowid)

    def get_messages(self, session_id: str, limit: Optional[int] = None) -> List[Dict]:
        """يرجع الرسائل بالترتيب الزمني. limit يرجع آخر N رسالة فقط."""
        with self._connect() as conn:
            if limit:
                rows = conn.execute(
                    "SELECT * FROM (SELECT * FROM messages WHERE session_id=? ORDER BY id DESC LIMIT ?) "
                    "ORDER BY id ASC",
                    (session_id, limit),
                ).fetchall()
            else:
                rows = conn.execute(
                    "SELECT * FROM messages WHERE session_id=? ORDER BY id ASC", (session_id,)
                ).fetchall()
        return [dict(r) for r in rows]

    def clear_messages(self, session_id: str) -> None:
        with self._connect() as conn:
            conn.execute("DELETE FROM messages WHERE session_id=?", (session_id,))

    # ------------------------------------------------------------- الترشيحات
    def save_recommendations(self, session_id: str, query: str, items: List[Dict], source: str = "llm") -> None:
        with self._connect() as conn:
            conn.executemany(
                "INSERT INTO recommendations (session_id, query, restaurant_id, restaurant_name, reason, "
                "match_score, source, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                [
                    (session_id, query, int(i["restaurant_id"]), i["name"], i.get("reason", ""),
                     int(i.get("match_score", 0)), source, _now())
                    for i in items
                ],
            )

    def get_history(self, session_id: str, limit: int = 50) -> List[Dict]:
        with self._connect() as conn:
            rows = conn.execute(
                "SELECT query, restaurant_name, reason, match_score, source, created_at "
                "FROM recommendations WHERE session_id=? ORDER BY id DESC LIMIT ?",
                (session_id, limit),
            ).fetchall()
        return [dict(r) for r in rows]

    # ------------------------------------------------------------- المفضلة
    def add_favorite(self, session_id: str, restaurant_id: int, name: str) -> None:
        with self._connect() as conn:
            conn.execute(
                "INSERT OR IGNORE INTO favorites (session_id, restaurant_id, restaurant_name, created_at) "
                "VALUES (?, ?, ?, ?)",
                (session_id, int(restaurant_id), name, _now()),
            )

    def remove_favorite(self, session_id: str, restaurant_id: int) -> None:
        with self._connect() as conn:
            conn.execute(
                "DELETE FROM favorites WHERE session_id=? AND restaurant_id=?", (session_id, int(restaurant_id))
            )

    def get_favorites(self, session_id: str) -> List[Dict]:
        with self._connect() as conn:
            rows = conn.execute(
                "SELECT restaurant_id, restaurant_name, created_at FROM favorites "
                "WHERE session_id=? ORDER BY created_at DESC",
                (session_id,),
            ).fetchall()
        return [dict(r) for r in rows]

    # ------------------------------------------------------------- التقييم
    def add_feedback(self, session_id: str, query: str, is_helpful: bool) -> None:
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO feedback (session_id, query, is_helpful, created_at) VALUES (?, ?, ?, ?)",
                (session_id, query, int(is_helpful), _now()),
            )

    def get_stats(self) -> Dict:
        """إحصائيات بسيطة لصفحة لوحة المعلومات."""
        with self._connect() as conn:
            total_q = conn.execute("SELECT COUNT(*) FROM messages WHERE role='user'").fetchone()[0]
            total_r = conn.execute("SELECT COUNT(*) FROM recommendations").fetchone()[0]
            helpful = conn.execute("SELECT COUNT(*) FROM feedback WHERE is_helpful=1").fetchone()[0]
            fb_total = conn.execute("SELECT COUNT(*) FROM feedback").fetchone()[0]
            top = conn.execute(
                "SELECT restaurant_name, COUNT(*) AS n FROM recommendations "
                "GROUP BY restaurant_name ORDER BY n DESC LIMIT 5"
            ).fetchall()
        return {
            "total_questions": total_q,
            "total_recommendations": total_r,
            "helpful_ratio": round(helpful / fb_total * 100) if fb_total else None,
            "top_restaurants": [dict(r) for r in top],
        }
''')
_w('src/memory.py', r'''"""
memory.py — ذاكرة المحادثة Conversation Memory (المسؤولة: لمى الأسمري)

نربط LangChain بقاعدة SQLite عن طريق كلاس يرث من BaseChatMessageHistory.
بهذا الشكل سجل المحادثة محفوظ في قاعدة البيانات، ونمرر آخر N رسالة للنموذج
عن طريق MessagesPlaceholder("history") في الـ Prompt.
"""
from __future__ import annotations

from typing import List, Sequence

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage

from .database import Database


class SQLiteChatHistory(BaseChatMessageHistory):
    """سجل محادثة LangChain مخزّن في جدول messages."""

    def __init__(self, db: Database, session_id: str, window: int = 6):
        self.db = db
        self.session_id = session_id
        self.window = window  # عدد الرسائل التي تُمرر للنموذج (نافذة الذاكرة)

    @property
    def messages(self) -> List[BaseMessage]:  # type: ignore[override]
        rows = self.db.get_messages(self.session_id, limit=self.window)
        result: List[BaseMessage] = []
        for r in rows:
            if r["role"] == "user":
                result.append(HumanMessage(content=r["content"]))
            else:
                result.append(AIMessage(content=r["content"]))
        return result

    def add_messages(self, messages: Sequence[BaseMessage]) -> None:
        for m in messages:
            role = "user" if isinstance(m, HumanMessage) else "assistant"
            self.db.add_message(self.session_id, role, str(m.content))

    def clear(self) -> None:
        self.db.clear_messages(self.session_id)
''')
_w('src/fallback.py', r'''"""
fallback.py — البحث المحلي الاحتياطي (المسؤولة: لمى الأسمري)

يُستخدم فقط عندما:
- لا يوجد مفتاح OpenAI، أو
- تعطل الاتصال بالنموذج اللغوي، أو
- رجع النموذج ردًا غير صالح.

يعتمد على قواعد بسيطة (كلمات مفتاحية) وفلترة مباشرة على جدول المطاعم،
حتى لا يتوقف النظام أثناء العرض. هذا ليس بديلًا عن LangChain بل خطة طوارئ.
"""
from __future__ import annotations

import re
from typing import Dict, List

import pandas as pd

PRICE_WORDS = {
    1: ["رخيص", "رخيصة", "اقتصادي", "اقتصادية", "طلاب", "طالب", "ميزانية محدودة"],
    2: ["متوسط", "معقول", "مناسب السعر"],
    3: ["غالي", "مرتفع"],
    4: ["فاخر", "فخم", "راقي", "راقية", "مناسبة خاصة"],
}

CUISINE_SYNONYMS = {
    "مأكولات بحرية": ["بحري", "بحرية", "سمك", "روبيان", "سي فود"],
    "سعودي": ["سعودي", "شعبي", "كبسة", "مندي", "مظبي", "حجازي", "نجدي"],
    "ياباني": ["ياباني", "سوشي", "رامن"],
    "إيطالي": ["إيطالي", "ايطالي", "بيتزا", "باستا"],
    "صحي": ["صحي", "دايت", "سعرات", "رجيم"],
    "مقهى": ["مقهى", "كافيه", "قهوة", "كوفي", "مذاكرة"],
    "حلويات": ["حلويات", "حلا", "كنافة", "بانكيك"],
    "شامي": ["شامي", "شاورما", "سوري"],
    "لبناني": ["لبناني"],
    "هندي": ["هندي", "برياني", "كاري"],
    "ستيك": ["ستيك", "ستيك هاوس"],
    "أمريكي": ["برجر", "أمريكي", "امريكي"],
    "تركي": ["تركي", "كباب"],
    "صيني": ["صيني", "نودلز"],
    "كوري": ["كوري"],
    "تايلندي": ["تايلندي"],
    "مكسيكي": ["مكسيكي", "تاكو"],
    "يمني": ["يمني", "حنيذ"],
    "مصري": ["مصري", "كشري", "فول", "طعمية"],
    "فرنسي": ["فرنسي"],
    "إيراني": ["إيراني", "ايراني", "فارسي"],
    "أفغاني": ["أفغاني", "افغاني", "بخاري"],
}

CITY_SYNONYMS = {
    "مكة المكرمة": ["مكة", "مكه"],
    "المدينة المنورة": ["المدينة", "المدينه"],
    "الرياض": ["الرياض", "رياض"],
    "جدة": ["جدة", "جده"],
    "الدمام": ["الدمام"],
    "الخبر": ["الخبر"],
    "أبها": ["أبها", "ابها"],
    "الطائف": ["الطائف"],
    "الأحساء": ["الأحساء", "الاحساء", "الهفوف"],
}


def extract_preferences_rule_based(text: str) -> Dict:
    """استخراج بسيط للتفضيلات بالكلمات المفتاحية (بدون نموذج لغوي)."""
    prefs: Dict = {}
    for city, words in CITY_SYNONYMS.items():
        if any(w in text for w in words):
            prefs["city"] = city
            break
    for cuisine, words in CUISINE_SYNONYMS.items():
        if any(w in text for w in words):
            prefs["cuisine"] = cuisine
            break
    for level, words in PRICE_WORDS.items():
        if any(w in text for w in words):
            prefs["max_price_level"] = level
            break
    budget = re.search(r"(\d{2,4})\s*(ريال|ر\.س|SAR)", text)
    if budget:
        prefs["max_budget_sar"] = int(budget.group(1))
    if any(w in text for w in ["عائل", "عايل", "أطفال", "اطفال", "عيال"]):
        prefs["family_friendly"] = True
    if any(w in text for w in ["نباتي", "نباتية", "فيجن"]):
        prefs["vegetarian"] = True
    if any(w in text for w in ["توصيل", "ديليفري", "يوصل"]):
        prefs["delivery"] = True
    if any(w in text for w in ["خارجي", "خارجية", "برا", "إطلالة", "اطلالة"]):
        prefs["outdoor_seating"] = True
    return prefs


def _tokens(text: str) -> List[str]:
    return [t for t in re.split(r"[\s،,.؟?!]+", text) if len(t) > 2]


def local_search(df: pd.DataFrame, query: str, prefs: Dict | None = None, top_n: int = 3) -> List[Dict]:
    """يرجع أفضل المطاعم حسب نقاط بسيطة: تطابق الشروط + كلمات الطلب + التقييم."""
    prefs = prefs if prefs is not None else extract_preferences_rule_based(query)
    words = _tokens(query)
    results = []
    for _, r in df.iterrows():
        score = 0.0
        if prefs.get("city"):
            score += 40 if r["city"] == prefs["city"] else -100
        if prefs.get("cuisine"):
            score += 30 if r["cuisine"] == prefs["cuisine"] else -10
        if prefs.get("max_price_level"):
            score += 10 if r["price_level"] <= prefs["max_price_level"] else -15 * (r["price_level"] - prefs["max_price_level"])
        if prefs.get("max_budget_sar"):
            score += 10 if r["avg_price_sar"] <= prefs["max_budget_sar"] else -60
        for flag, col in [("family_friendly", "family_friendly"), ("vegetarian", "vegetarian_options"),
                          ("delivery", "delivery"), ("outdoor_seating", "outdoor_seating")]:
            if prefs.get(flag):
                score += 8 if r[col] else -15
        haystack = f"{r['name']} {r['cuisine']} {r['specialties']} {r['description']}"
        score += sum(3 for w in words if w in haystack)
        score += float(r["rating"]) * 2
        results.append((score, r))

    results.sort(key=lambda x: x[0], reverse=True)
    best = results[:top_n]
    max_score = max((s for s, _ in best), default=1) or 1
    output = []
    for s, r in best:
        output.append({
            "restaurant_id": int(r["id"]),
            "name": r["name"],
            "reason": f"{r['cuisine']} في {r['city']} - {r['district']}، تقييمه {r['rating']}، "
                      f"ومتوسط السعر {r['avg_price_sar']} ريال للشخص.",
            "suggested_dishes": [d.strip() for d in str(r["specialties"]).split("؛")][:2],
            "match_score": max(0, min(100, int(s / max_score * 85))),
        })
    return output
''')

### 3.1 إنشاء قاعدة البيانات والتحقق من الجداول

In [ ]:
import sqlite3
from src.database import Database

db = Database("restaurant_app.db")
with sqlite3.connect("restaurant_app.db") as conn:
    tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]
print("الجداول:", tables)

### 3.2 حفظ واسترجاع (رسائل، ترشيحات، مفضلة، تقييم)

In [ ]:
sid = db.create_session()
db.add_message(sid, "user", "أبي مطعم بحري في جدة")
db.add_message(sid, "assistant", "أرشح لك لؤلؤة البحر")
db.save_recommendations(sid, "أبي مطعم بحري في جدة",
                        [{"restaurant_id": 9, "name": "لؤلؤة البحر", "reason": "بحري على الكورنيش", "match_score": 92}])
db.add_favorite(sid, 9, "لؤلؤة البحر"); db.add_favorite(sid, 9, "لؤلؤة البحر")  # المرة الثانية ما تتكرر
db.add_feedback(sid, "أبي مطعم بحري في جدة", True)

print("الرسائل:", [(m["role"], m["content"]) for m in db.get_messages(sid)])
print("الترشيحات:", db.get_history(sid))
print("المفضلة (بدون تكرار):", db.get_favorites(sid))
print("الإحصائيات:", db.get_stats())

### 3.3 ذاكرة المحادثة (آخر N رسائل)

In [ ]:
from src.memory import SQLiteChatHistory
history = SQLiteChatHistory(db, sid, window=6)
for m in history.messages:
    print(type(m).__name__, "→", m.content)

### 3.4 البحث المحلي الاحتياطي (يشتغل بدون مفتاح)

In [ ]:
from src.fallback import extract_preferences_rule_based, local_search
q = "أبي مطعم بحري رخيص في جده للعائلة"
print("التفضيلات:", extract_preferences_rule_based(q))
for r in local_search(df, q):
    print(f"  - {r['name']} ({r['match_score']}%) — {r['reason']}")

### 3.5 اختبار قاعدة بيانات غير متاحة

In [ ]:
from src.errors import DatabaseUnavailableError
try:
    Database("/مجلد_غير_موجود/x.db")
    print("FAIL")
except DatabaseUnavailableError as error:
    print("PASS |", error.user_message)

---
# 4) جزئية عائشة المالكي — هندسة الأوامر وبناء السلاسل
1. Prompt-ان: استخراج التفضيلات + الترشيح (دور، قواعد، أمثلة Few-shot، ذاكرة)
2. مخرجات منظمة بـ Pydantic (`with_structured_output`)
3. `RecommendationEngine`: يربط كل أجزاء الفريق + Guardrail ضد الهلوسة

In [ ]:
#@title prompts.py · chain.py (الكود مخفي) { display-mode: "form" }
from pathlib import Path

def _w(path, text):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8'); print('✓', path)

_w('src/prompts.py', r'''"""
prompts.py — تصميم الـ Prompts (المسؤولة: عائشة المالكي)

عندنا Prompt-ان:
1) EXTRACTION_PROMPT: يحوّل كلام المستخدم الحر إلى تفضيلات منظمة.
2) RECOMMENDATION_PROMPT: يختار أفضل المطاعم من السياق المسترجع فقط ويشرح السبب.

أساليب الـ Prompt Engineering المستخدمة:
- تحديد الدور (Role) والجمهور واللغة.
- قواعد صريحة تمنع اختراع مطاعم غير موجودة (Grounding).
- أمثلة (Few-shot) لتوضيح الاستخراج.
- تمرير سجل المحادثة (Memory) لفهم طلبات المتابعة مثل "أرخص منه".
- إخراج منظم (Structured Output) بدل النص الحر.
"""
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ---------------------------------------------------------------- 1) استخراج التفضيلات
EXTRACTION_SYSTEM = """أنت محلل طلبات في نظام ترشيح مطاعم سعودي.
مهمتك: استخراج تفضيلات المستخدم من رسالته الأخيرة بشكل منظم.

القواعد:
- استخدم سجل المحادثة لفهم طلبات المتابعة. مثال: إذا طلب سابقًا مطاعم في جدة ثم قال "أبي أرخص" فالمدينة ما زالت جدة.
- اكتب المدينة ونوع المطبخ بالعربي وبنفس الأسماء المتاحة:
  المدن: {cities}
  المطابخ: {cuisines}
- مستوى السعر: رخيص/اقتصادي = 1، متوسط = 2، غالي/مرتفع = 3، فاخر/راقي = 4.
- إذا لم يذكر المستخدم معلومة فاتركها فارغة (null) ولا تخمّن.
- إذا كان الطلب لا علاقة له بالأكل أو المطاعم اجعل is_food_related = false.

أمثلة:
- "أبي مطعم بحري في جدة للعائلة" ← city=جدة، cuisine=مأكولات بحرية، family_friendly=true
- "مكان رخيص للفطور في الرياض" ← city=الرياض، max_price_level=1، occasion=فطور
- "عشاء عمل فاخر ستيك" ← cuisine=ستيك، max_price_level=4، occasion=عشاء عمل"""

EXTRACTION_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", EXTRACTION_SYSTEM),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)

# ---------------------------------------------------------------- 2) الترشيح النهائي
RECOMMENDATION_SYSTEM = """أنت "مُرشِد"، مساعد ذكي لترشيح المطاعم في السعودية. تتحدث بالعربية بأسلوب ودود ومختصر.

القواعد المهمة:
1. رشّح فقط من المطاعم الموجودة في "المطاعم المتاحة" أدناه. ممنوع اختراع مطعم أو معلومة غير موجودة.
2. اختر من 1 إلى 3 مطاعم الأنسب لطلب المستخدم، ورتبها من الأنسب للأقل.
3. اذكر لكل مطعم سببًا واضحًا يربط بين طلب المستخدم وبيانات المطعم (السعر، الموقع، الأطباق، العائلة...).
4. اكتب restaurant_id والاسم كما وردا في السياق حرفيًا.
5. إذا لا يوجد مطعم مناسب تمامًا، رشّح الأقرب وقل ذلك بصراحة في الملخص.
6. إذا كان الطلب لا علاقة له بالمطاعم، اعتذر بلطف في الملخص واترك القائمة فارغة.
7. لا تذكر أسعارًا أو تقييمات غير موجودة في السياق.

المطاعم المتاحة (نتائج البحث من قاعدة المتجهات):
{context}

التفضيلات المستخرجة من طلب المستخدم:
{preferences}"""

RECOMMENDATION_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", RECOMMENDATION_SYSTEM),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)
''')
_w('src/chain.py', r'''"""
chain.py — قلب النظام: سلسلة LangChain الكاملة (المسؤولة: عائشة المالكي)

مسار الطلب (Pipeline):
  رسالة المستخدم
    → (1) التحقق من المدخلات
    → (2) Extraction Chain: Prompt | LLM.with_structured_output(UserPreferences)
    → (3) Retrieval: Chroma Retriever مع فلترة metadata وتخفيف تدريجي
    → (4) Recommendation Chain: Prompt | LLM.with_structured_output(RecommendationResponse)
    → (5) Guardrail: التأكد أن كل مطعم مرشّح موجود فعلًا في نتائج البحث
    → (6) الحفظ في SQLite (الرسائل + الترشيحات)
وعند أي خطأ في OpenAI ننتقل إلى البحث المحلي (fallback.py).
"""
from __future__ import annotations

import logging
import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from .errors import (
    AppError,
    InvalidModelResponseError,
    InvalidUserInputError,
    LLMUnavailableError,
    MissingAPIKeyError,
)
from .fallback import extract_preferences_rule_based, local_search
from .schemas import RecommendationResponse, UserPreferences

logger = logging.getLogger(__name__)

MIN_LEN, MAX_LEN = 3, 500


@dataclass
class EngineResult:
    """النتيجة التي تعرضها الواجهة."""

    summary: str
    recommendations: List[Dict] = field(default_factory=list)
    follow_up_question: Optional[str] = None
    preferences: Dict = field(default_factory=dict)
    source: str = "llm"            # llm أو local
    warning: Optional[str] = None  # رسالة خطأ ودية إن وُجدت
    relax_level: int = 0           # كم مرة خففنا الفلاتر


def validate_user_input(text: str) -> str:
    """Error Handling للإدخال الخاطئ: فارغ، قصير جدًا، طويل جدًا، أو رموز فقط."""
    text = (text or "").strip()
    if len(text) < MIN_LEN:
        raise InvalidUserInputError("الطلب قصير جدًا")
    if len(text) > MAX_LEN:
        raise InvalidUserInputError("الطلب أطول من 500 حرف")
    if not re.search(r"[ء-يA-Za-z]", text):  # حروف عربية أو إنجليزية فعلية
        raise InvalidUserInputError("الطلب لا يحتوي على كلمات")
    return text


class RecommendationEngine:
    """يجمع كل مكونات LangChain في كلاس واحد تستخدمه الواجهة."""

    def __init__(self, settings, df, db):
        self.settings = settings
        self.df = df
        self.db = db
        self.store = None
        self.llm = None
        self.init_error: Optional[AppError] = None
        self._last_prefs: Dict[str, Dict] = {}  # ذاكرة بسيطة للبحث المحلي: آخر تفضيلات لكل جلسة
        self._last_top_price: Dict[str, int] = {}
        self._setup()

    # ------------------------------------------------------------------ الإعداد
    def _setup(self) -> None:
        if not self.settings.has_api_key:
            self.init_error = MissingAPIKeyError()
            return
        try:
            from langchain_openai import ChatOpenAI

            from .vector_store import build_or_load_vector_store

            self.llm = ChatOpenAI(
                model=self.settings.chat_model,
                temperature=self.settings.temperature,
                api_key=self.settings.openai_api_key,
                timeout=30,
                max_retries=2,
            )
            self.store = build_or_load_vector_store(self.df, self.settings)
        except Exception as exc:  # مفتاح خاطئ، لا يوجد إنترنت، حصة منتهية...
            logger.exception("فشل تهيئة OpenAI/Chroma")
            self.init_error = LLMUnavailableError(str(exc))
            self.llm, self.store = None, None

    @property
    def llm_ready(self) -> bool:
        return self.llm is not None and self.store is not None

    # ------------------------------------------------------------------ السلاسل
    def _extraction_chain(self):
        from .prompts import EXTRACTION_PROMPT

        return EXTRACTION_PROMPT | self.llm.with_structured_output(UserPreferences)

    def _recommendation_chain(self):
        from .prompts import RECOMMENDATION_PROMPT

        return RECOMMENDATION_PROMPT | self.llm.with_structured_output(RecommendationResponse)

    # ------------------------------------------------------------------ التشغيل
    def recommend(self, question: str, session_id: str) -> EngineResult:
        question = validate_user_input(question)  # يرمي InvalidUserInputError للواجهة

        history = self._history(session_id)
        if self.llm_ready:
            try:
                result = self._run_llm(question, history)
            except InvalidModelResponseError as exc:
                result = self._run_local(question, warning=exc.user_message, session_id=session_id)
            except Exception as exc:
                logger.exception("خطأ أثناء استدعاء النموذج")
                result = self._run_local(question, warning=LLMUnavailableError(str(exc)).user_message,
                                         session_id=session_id)
        else:
            warning = self.init_error.user_message if self.init_error else None
            result = self._run_local(question, warning=warning, session_id=session_id)

        self._save(session_id, question, result)
        return result

    def _history(self, session_id: str):
        try:
            from .memory import SQLiteChatHistory

            return SQLiteChatHistory(self.db, session_id, self.settings.history_window).messages
        except Exception:
            return []  # إذا تعطلت القاعدة نكمل بدون ذاكرة

    def _run_llm(self, question: str, history) -> EngineResult:
        from .vector_store import format_docs, retrieve_restaurants

        cities = sorted(self.df["city"].unique())
        cuisines = sorted(self.df["cuisine"].unique())

        # (2) استخراج التفضيلات
        prefs: UserPreferences = self._extraction_chain().invoke(
            {"question": question, "history": history,
             "cities": "، ".join(cities), "cuisines": "، ".join(cuisines)}
        )
        if prefs is None:
            raise InvalidModelResponseError("لم يرجع النموذج تفضيلات")

        # نتجاهل قيمًا غير موجودة في البيانات بدل أن تفشل الفلترة
        if prefs.city and prefs.city not in cities:
            prefs.city = None
        if prefs.cuisine and prefs.cuisine not in cuisines:
            prefs.cuisine = None

        if not prefs.is_food_related:
            return EngineResult(
                summary="أنا متخصص في ترشيح المطاعم فقط 😊 قل لي وش تشتهي، وفي أي مدينة، وكم ميزانيتك.",
                preferences=prefs.model_dump(exclude_none=True),
            )

        # (3) Retrieval
        docs, relax_level = retrieve_restaurants(self.store, prefs, k=self.settings.top_k)
        if not docs:
            raise InvalidModelResponseError("لا توجد نتائج من البحث")

        # (4) الترشيح
        response: RecommendationResponse = self._recommendation_chain().invoke(
            {"question": question, "history": history,
             "context": format_docs(docs),
             "preferences": prefs.model_dump_json(exclude_none=True)}
        )

        # (5) Guardrail ضد الهلوسة: نقبل فقط المطاعم الموجودة في نتائج البحث
        valid_ids = {int(d.metadata["id"]) for d in docs}
        by_id = {int(d.metadata["id"]): d.metadata for d in docs}
        recs = []
        for r in (response.recommendations if response else []):
            if r.restaurant_id in valid_ids:
                item = r.model_dump()
                item["name"] = by_id[r.restaurant_id]["name"]  # الاسم الصحيح من البيانات
                recs.append(item)
        if response is None or (response.recommendations and not recs):
            raise InvalidModelResponseError("النموذج رشّح مطاعم غير موجودة")

        warning = None
        if relax_level > 0:
            warning = "ما لقينا تطابقًا كاملًا لكل الشروط، فخففنا بعضها وعرضنا الأقرب لطلبك."
        return EngineResult(
            summary=response.summary,
            recommendations=recs,
            follow_up_question=response.follow_up_question,
            preferences=prefs.model_dump(exclude_none=True),
            source="llm",
            warning=warning,
            relax_level=relax_level,
        )

    def _run_local(self, question: str, warning: Optional[str] = None, session_id: str = "") -> EngineResult:
        new = extract_preferences_rule_based(question)
        last = self._last_prefs.get(session_id, {})
        # طلبات المتابعة: نكمل على التفضيلات السابقة ونعدّل الجديد فقط
        prefs = {**last, **new} if last and len(new) <= 1 else new
        if any(w in question for w in ["أرخص", "ارخص"]) and self._last_top_price.get(session_id):
            # «أرخص»: ميزانية أقل من سعر الترشيح الأول السابق
            prefs["max_budget_sar"] = self._last_top_price[session_id] - 1
        self._last_prefs[session_id] = prefs
        recs = local_search(self.df, question, prefs)
        if recs:
            top = self.df.loc[self.df["id"] == recs[0]["restaurant_id"], "avg_price_sar"]
            if not top.empty:
                self._last_top_price[session_id] = int(top.iloc[0])
        return EngineResult(
            summary="هذه أقرب المطاعم لطلبك من البحث المحلي:",
            recommendations=recs,
            preferences=prefs,
            source="local",
            warning=warning,
        )

    def _save(self, session_id: str, question: str, result: EngineResult) -> None:
        """يحفظ السؤال والرد والترشيحات. فشل الحفظ لا يوقف عرض النتيجة."""
        try:
            names = "، ".join(r["name"] for r in result.recommendations)
            self.db.add_message(session_id, "user", question)
            self.db.add_message(session_id, "assistant", f"{result.summary} المطاعم: {names}".strip())
            if result.recommendations:
                self.db.save_recommendations(session_id, question, result.recommendations, result.source)
        except AppError as exc:
            result.warning = (result.warning + " " if result.warning else "") + exc.user_message
''')

### 4.1 شكل الـ Prompt الجاهز للنموذج

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from src.prompts import EXTRACTION_PROMPT

value = EXTRACTION_PROMPT.invoke({
    "question": "طيب أبي أرخص",
    "history": [HumanMessage("أبي مطعم بحري في جدة"), AIMessage("أرشح لك لؤلؤة البحر")],
    "cities": "، ".join(sorted(df.city.unique())), "cuisines": "، ".join(sorted(df.cuisine.unique())),
})
print(value.to_string())

### 4.2 اختبار التحقق من المدخلات

In [ ]:
from src.chain import validate_user_input
from src.errors import InvalidUserInputError
for bad in ["", "؟؟؟؟", "!!!!!", "123456", "a" * 501]:
    try:
        validate_user_input(bad); print("FAIL |", repr(bad[:10]))
    except InvalidUserInputError as error:
        print("PASS |", repr(bad[:10]), "→", error.detail)
print("مقبول ✓:", validate_user_input("  مطعم في جدة  "))

### 4.3 سلسلة الاستخراج (Extraction Chain)

In [ ]:
from src.chain import RecommendationEngine
engine = RecommendationEngine(get_settings(), df, db)
print("OpenAI متصل ✓" if engine.llm_ready else f"وضع البحث المحلي: {engine.init_error.user_message if engine.init_error else ''}")

if engine.llm_ready:
    chain = engine._extraction_chain()
    for q in ["أبي مطعم بحري في جدة مناسب للعائلة وسعره متوسط", "عشاء عمل فاخر ستيك في الرياض", "وش عاصمة فرنسا؟"]:
        p = chain.invoke({"question": q, "history": [], "cities": "، ".join(sorted(df.city.unique())),
                          "cuisines": "، ".join(sorted(df.cuisine.unique()))})
        print(f"\n🧑 {q}\n   → {p.model_dump(exclude_none=True)}")

### 4.4 النظام الكامل: ترشيح + طلب متابعة (الذاكرة)

In [ ]:
session = db.create_session()
for q in ["أبي مطعم بحري في جدة مناسب للعائلة وسعره متوسط", "طيب أبي أرخص"]:
    r = engine.recommend(q, session)
    print(f"\n🧑 {q}")
    print("   المصدر:", r.source, "| فهمت طلبك:", r.preferences)
    if r.warning: print("   ⚠️", r.warning)
    print("   🍽️", r.summary)
    for i, rec in enumerate(r.recommendations, 1):
        print(f"     {i}. {rec['name']} ({rec['match_score']}%) — {rec['reason']}")

---
# 5) جزئية رنيم حكمي — واجهة المستخدم Streamlit
1. التصميم في فيقما أولًا (ألوان الشعار + خط Tajawal + اتجاه RTL)
2. خمس صفحات: المحادثة · المفضلة · سجل الترشيحات · لوحة المعلومات · عن المشروع
3. بطاقة لكل مطعم: الترتيب، نسبة التطابق، السعر، السبب، الأطباق، زر المفضلة

> أوامر Streamlit ما تنعرض داخل الدفتر، لذلك تُكتب الواجهة في `app.py` ثم تشتغل برابط في الأخير.

In [ ]:
#@title الشعار وخط Tajawal (الكود مخفي) { display-mode: "form" }
import base64, urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True); Path("static").mkdir(exist_ok=True)
Path("assets/university_logo.png").write_bytes(base64.b64decode("..."))
print("✓ assets/university_logo.png")
for style in ["Regular", "Medium", "Bold", "ExtraBold"]:
    target = Path(f"static/Tajawal-{style}.ttf")
    if not target.exists():
        try:
            urllib.request.urlretrieve(f"https://github.com/google/fonts/raw/main/ofl/tajawal/Tajawal-{style}.ttf", target)
            print("✓", target)
        except Exception as error:
            print("ما تحمّل الخط", style, "— بيستخدم خط بديل:", error)

In [ ]:
#@title styles.py · components.py · app.py (الكود مخفي) { display-mode: "form" }
from pathlib import Path

def _w(path, text):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8'); print('✓', path)

_w('src/ui/__init__.py', r'''''')
_w('src/ui/styles.py', r'''"""
styles.py — تنسيق الواجهة CSS (المسؤولة: رنيم حكمي)

الألوان والخطوط مأخوذة من تصميم فيقما (Design Tokens):
كحلي فيروزي #0F3D4C، فيروزي #12808A، بنفسجي الشعار #5B4B9A، زعفراني #E3A33A، خلفية كريمية #F7F4EE.
الخط: Tajawal محمّل من مجلد static (يحتاج enableStaticServing في .streamlit/config.toml).
"""

COLORS = {
    "deep": "#0F3D4C",
    "teal": "#12808A",
    "teal_light": "#E3F2F1",
    "purple": "#5B4B9A",
    "saffron": "#E3A33A",
    "saffron_light": "#FCF1DE",
    "bg": "#F7F4EE",
    "ink": "#1D2A32",
    "muted": "#6B7780",
    "line": "#E8E2D6",
}

GLOBAL_CSS = f"""
<style>
@font-face {{ font-family: 'Tajawal'; src: url('app/static/Tajawal-Regular.ttf') format('truetype'); font-weight: 400; }}
@font-face {{ font-family: 'Tajawal'; src: url('app/static/Tajawal-Medium.ttf') format('truetype'); font-weight: 500; }}
@font-face {{ font-family: 'Tajawal'; src: url('app/static/Tajawal-Bold.ttf') format('truetype'); font-weight: 700; }}
@font-face {{ font-family: 'Tajawal'; src: url('app/static/Tajawal-ExtraBold.ttf') format('truetype'); font-weight: 800; }}

html, body, [class*="css"], .stApp, .stMarkdown, button, input, textarea, select {{
    font-family: 'Tajawal', sans-serif !important;
}}
.stApp {{ direction: rtl; background: {COLORS['bg']}; color: {COLORS['ink']}; }}
.stMarkdown, .stText, p, h1, h2, h3, h4, label {{ text-align: right; }}
/* إخفاء عناصر Streamlit الافتراضية غير المهمة */
#MainMenu, footer {{ visibility: hidden; }}
.block-container {{ padding-top: 1.6rem; max-width: 1180px; }}

/* ------------------------------------------------ الشريط الجانبي */
section[data-testid="stSidebar"] {{ background: {COLORS['deep']}; }}
section[data-testid="stSidebar"] * {{ color: #FFFFFFDD; }}
section[data-testid="stSidebar"] .side-title {{ font-size: 1.1rem; font-weight: 700; color: #fff; margin: .6rem 0 0; }}
section[data-testid="stSidebar"] .side-sub {{ font-size: .82rem; opacity: .65; }}
.logo-box {{ background: #fff; border-radius: 18px; padding: 14px; text-align: center; }}
.logo-box img {{ max-width: 100%; }}
.status-box {{ background: rgba(255,255,255,.07); border-radius: 14px; padding: 12px 14px; font-size: .85rem; line-height: 1.9; }}
.status-box b {{ color: {COLORS['saffron']} !important; }}
.team {{ font-size: .82rem; line-height: 1.8; opacity: .85; }}
.team b {{ color: {COLORS['saffron']} !important; }}

/* ------------------------------------------------ الهيدر */
.hero {{
    background: linear-gradient(100deg, {COLORS['deep']} 0%, {COLORS['teal']} 50%, {COLORS['purple']} 100%);
    border-radius: 24px; padding: 26px 30px; color: #fff;
    display: flex; justify-content: space-between; align-items: center; gap: 16px; margin-bottom: 14px;
}}
.hero h1 {{ color: #fff; font-size: 2.4rem; font-weight: 800; margin: 0; padding: 0; line-height: 1.2; }}
.hero p {{ color: #ffffffE0; margin: 4px 0 0; font-size: 1rem; font-weight: 500; }}
.pill-status {{ background: rgba(255,255,255,.16); border-radius: 999px; padding: 8px 14px; font-size: .82rem; white-space: nowrap; direction: ltr; }}
.dot {{ display: inline-block; width: 8px; height: 8px; border-radius: 50%; margin-inline-start: 6px; }}

/* ------------------------------------------------ الشرائح الصغيرة Chips */
.chips {{ display: flex; flex-wrap: wrap; gap: 8px; align-items: center; margin: 6px 0 10px; }}
.chip {{ background: {COLORS['teal_light']}; color: {COLORS['teal']}; border-radius: 999px; padding: 4px 12px; font-size: .82rem; font-weight: 500; }}
.chip.dish {{ background: {COLORS['saffron_light']}; color: #9A6412; }}
.chip-label {{ color: {COLORS['muted']}; font-weight: 700; font-size: .88rem; }}
.src-badge {{ background: #EEEAF7; color: {COLORS['purple']}; border-radius: 8px; padding: 3px 10px; font-size: .75rem; font-weight: 700; direction: ltr; display: inline-block; }}
.src-badge.local {{ background: {COLORS['saffron_light']}; color: #9A6412; }}

/* ------------------------------------------------ بطاقة المطعم */
.r-card {{
    background: #fff; border: 1px solid {COLORS['line']}; border-radius: 20px; padding: 18px 18px 12px;
    box-shadow: 0 8px 24px rgba(15,61,76,.08); height: 100%;
}}
.r-top {{ display: flex; justify-content: space-between; align-items: center; }}
.r-rank {{ background: {COLORS['saffron_light']}; color: #9A6412; border-radius: 999px; padding: 3px 10px; font-size: .75rem; font-weight: 700; }}
.r-match {{ color: {COLORS['teal']}; font-weight: 700; font-size: .82rem; }}
.r-name {{ font-size: 1.35rem; font-weight: 800; margin: 10px 0 4px; color: {COLORS['ink']}; }}
.r-meta {{ color: {COLORS['muted']}; font-size: .88rem; display: flex; gap: 8px; align-items: center; flex-wrap: wrap; }}
.r-stats {{ display: flex; gap: 18px; margin: 12px 0 8px; }}
.r-stats div {{ display: flex; flex-direction: column; }}
.r-stats b {{ font-size: 1rem; }}
.r-stats span {{ font-size: .7rem; color: {COLORS['muted']}; }}
.r-bar {{ background: #ECE7DD; border-radius: 999px; height: 6px; overflow: hidden; margin: 6px 0 10px; }}
.r-bar > div {{ height: 6px; border-radius: 999px; background: linear-gradient(90deg, {COLORS['teal']}, {COLORS['purple']}); margin-inline-start: 0; }}
.r-reason {{ color: #46535C; font-size: .9rem; line-height: 1.7; }}

/* ------------------------------------------------ عناصر Streamlit */
.stButton > button {{
    border-radius: 12px; border: 1.5px solid {COLORS['teal']}; color: {COLORS['teal']};
    background: #fff; font-weight: 700; width: 100%;
}}
.stButton > button:hover {{ background: {COLORS['teal_light']}; border-color: {COLORS['teal']}; color: {COLORS['teal']}; }}
div[data-testid="stChatInput"] textarea {{ direction: rtl; text-align: right; }}
div[data-testid="stChatMessage"] {{ background: #fff; border: 1px solid {COLORS['line']}; border-radius: 20px; padding: 14px 18px; }}
.metric-card {{ background: #fff; border: 1px solid {COLORS['line']}; border-radius: 18px; padding: 18px; text-align: center; }}
.metric-card .v {{ font-size: 2rem; font-weight: 800; color: {COLORS['teal']}; }}
.metric-card .l {{ color: {COLORS['muted']}; font-size: .9rem; }}
</style>
"""
''')
_w('src/ui/components.py', r'''"""
components.py — مكونات الواجهة القابلة لإعادة الاستخدام (المسؤولة: رنيم حكمي)

كل دالة ترجع HTML لعنصر واحد من تصميم فيقما:
Hero (الهيدر)، Chips (التفضيلات والأطباق)، RestaurantCard (بطاقة المطعم).
نستخدم html.escape لحماية الواجهة من أي نص غريب قادم من النموذج أو من البيانات.
"""
from __future__ import annotations

import base64
from html import escape
from pathlib import Path
from typing import Dict, Iterable, Optional

from ..data_loader import PRICE_LABELS

PREF_LABELS = {
    "family_friendly": "مناسب للعائلة",
    "vegetarian": "خيارات نباتية",
    "delivery": "توصيل",
    "outdoor_seating": "جلسات خارجية",
}


def image_to_data_uri(path: Path) -> str:
    """يحوّل صورة الشعار إلى data URI لعرضها داخل HTML."""
    data = base64.b64encode(Path(path).read_bytes()).decode()
    return f"data:image/png;base64,{data}"


def hero_html(model_name: str, llm_ready: bool) -> str:
    status = f"متصل · OpenAI {escape(model_name)}" if llm_ready else "وضع البحث المحلي (بدون OpenAI)"
    color = "#7CF0B4" if llm_ready else "#F5C26B"
    return f"""
    <div class="hero">
      <div>
        <h1>مُرشِد</h1>
        <p>نظام ترشيح المطاعم الذكي — قل لي وش تشتهي، وفي أي مدينة، وكم ميزانيتك</p>
      </div>
      <div class="pill-status">{status}<span class="dot" style="background:{color}"></span></div>
    </div>"""


def chips_html(items: Iterable[str], label: Optional[str] = None, kind: str = "") -> str:
    chips = "".join(f'<span class="chip {kind}">{escape(str(i))}</span>' for i in items if i)
    lab = f'<span class="chip-label">{escape(label)}</span>' if label else ""
    return f'<div class="chips">{lab}{chips}</div>' if chips else ""


def preferences_to_labels(prefs: Dict) -> list:
    """يحوّل التفضيلات المستخرجة إلى كلمات قصيرة تُعرض للمستخدم (فهمت طلبك: ...)."""
    labels = []
    if prefs.get("city"):
        labels.append(prefs["city"])
    if prefs.get("cuisine"):
        labels.append(prefs["cuisine"])
    if prefs.get("max_price_level"):
        labels.append(f"سعر {PRICE_LABELS.get(int(prefs['max_price_level']), '')}")
    if prefs.get("max_budget_sar"):
        labels.append(f"حتى {prefs['max_budget_sar']} ريال")
    for key, lab in PREF_LABELS.items():
        if prefs.get(key):
            labels.append(lab)
    if prefs.get("occasion"):
        labels.append(prefs["occasion"])
    return labels


def restaurant_card_html(rank: int, rec: Dict, info: Optional[Dict]) -> str:
    """بطاقة مطعم واحدة. rec من النموذج، info من بيانات المطعم الأصلية."""
    info = info or {}
    score = int(rec.get("match_score", 0))
    price_level = int(info.get("price_level", 2))
    dishes = rec.get("suggested_dishes") or str(info.get("specialties", "")).split("؛")[:2]
    return f"""
    <div class="r-card">
      <div class="r-top">
        <span class="r-rank">الترشيح {rank}</span>
        <span class="r-match">تطابق {score}%</span>
      </div>
      <div class="r-name">{escape(rec.get('name', ''))}</div>
      <div class="r-meta">
        <span class="chip">{escape(str(info.get('cuisine', '')))}</span>
        <span>{escape(str(info.get('city', '')))} · {escape(str(info.get('district', '')))}</span>
      </div>
      <div class="r-stats">
        <div><b>{info.get('rating', '-')} / 5</b><span>التقييم</span></div>
        <div><b>{PRICE_LABELS.get(price_level, '-')}</b><span>مستوى السعر</span></div>
        <div><b>{info.get('avg_price_sar', '-')} ريال</b><span>للشخص</span></div>
      </div>
      <div class="r-bar"><div style="width:{max(4, min(score, 100))}%"></div></div>
      <div class="r-reason">{escape(rec.get('reason', ''))}</div>
      {chips_html([d.strip() for d in dishes], kind='dish')}
      <div class="r-meta" style="font-size:.78rem">أوقات العمل: {escape(str(info.get('opening_hours', '')))}</div>
    </div>"""


def metric_html(value, label: str) -> str:
    return f'<div class="metric-card"><div class="v">{escape(str(value))}</div><div class="l">{escape(label)}</div></div>'
''')
_w('app.py', r'''"""
app.py — واجهة المستخدم بـ Streamlit (المسؤولة: رنيم حكمي)

التشغيل:  streamlit run app.py

الصفحات:
1) المحادثة: يكتب المستخدم طلبه ويستلم 1-3 مطاعم مع السبب.
2) المفضلة: المطاعم التي حفظها المستخدم (من SQLite).
3) سجل الترشيحات: كل الترشيحات السابقة في الجلسة.
4) لوحة المعلومات: إحصائيات من قاعدة البيانات.
5) عن المشروع والبيانات: شرح المعمارية + رفع ملف مطاعم والتحقق منه.
"""
from __future__ import annotations

# ---------------------------------------------------------------------------
# Streamlit Cloud: Chroma يحتاج sqlite3 حديث. إذا توفرت pysqlite3 نستخدمها بدلًا من القديمة.
try:  # pragma: no cover
    __import__("pysqlite3")
    import sys

    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass
# ---------------------------------------------------------------------------

import dataclasses
from pathlib import Path

import pandas as pd
import streamlit as st

from src.chain import RecommendationEngine
from src.config import BASE_DIR, get_settings
from src.data_loader import PRICE_LABELS, load_restaurants
from src.database import Database
from src.errors import AppError, DatabaseUnavailableError, UnsupportedFileError
from src.ui.components import (
    chips_html,
    hero_html,
    image_to_data_uri,
    metric_html,
    preferences_to_labels,
    restaurant_card_html,
)
from src.ui.styles import GLOBAL_CSS

st.set_page_config(page_title="مُرشِد | نظام ترشيح المطاعم الذكي", page_icon="🍽️", layout="wide")
st.markdown(GLOBAL_CSS, unsafe_allow_html=True)

LOGO_PATH = BASE_DIR / "assets" / "university_logo.png"
SUGGESTIONS = [
    "مطعم بحري عائلي في جدة",
    "فطور شعبي رخيص في الرياض",
    "عشاء عمل فاخر ستيك",
    "مكان هادئ للمذاكرة في الرياض",
]
TEAM = ["يارا المالكي", "همس الشهري", "عائشة المالكي", "رنيم حكمي", "لمى الأسمري"]


# =========================================================================== تحميل المكونات
@st.cache_resource(show_spinner="جاري تجهيز قاعدة البيانات…")
def get_db(db_path: str):
    try:
        return Database(db_path)
    except DatabaseUnavailableError:
        return None


@st.cache_resource(show_spinner="جاري تحميل المطاعم وبناء قاعدة المتجهات…")
def get_engine(data_path: str, _db):
    settings = dataclasses.replace(get_settings(), data_path=Path(data_path))
    if Path(data_path).name != "restaurants.csv":
        # ملف مرفوع: نستخدم مجلد Chroma مستقل حتى لا نخرب الأصلي
        settings = dataclasses.replace(settings, chroma_dir=BASE_DIR / "chroma_db_uploaded")
    df = load_restaurants(data_path)
    return RecommendationEngine(settings, df, _db), df


settings = get_settings()
db = get_db(str(settings.db_path))

if "data_path" not in st.session_state:
    st.session_state.data_path = str(settings.data_path)

try:
    engine, df = get_engine(st.session_state.data_path, db)
except UnsupportedFileError as exc:
    st.error(f"{exc.user_message} ({exc.detail})")
    st.stop()

# جلسة المستخدم
if "session_id" not in st.session_state:
    try:
        st.session_state.session_id = db.create_session() if db else "offline"
    except AppError:
        st.session_state.session_id = "offline"
SESSION = st.session_state.session_id
st.session_state.setdefault("messages", [])      # [{role, content, result}]
st.session_state.setdefault("pending", None)      # اقتراح ضغطه المستخدم

restaurants_by_id = {int(r["id"]): r for r in df.to_dict("records")}


# =========================================================================== الشريط الجانبي
with st.sidebar:
    if LOGO_PATH.exists():
        st.markdown(
            f'<div class="logo-box"><img src="{image_to_data_uri(LOGO_PATH)}" alt="شعار الجامعة"/></div>',
            unsafe_allow_html=True,
        )
    st.markdown(
        '<div class="side-title">نظام ترشيح المطاعم الذكي</div>'
        '<div class="side-sub">مشروع مادة أساسيات LangChain</div>',
        unsafe_allow_html=True,
    )
    page = st.radio(
        "التنقل",
        ["المحادثة", "المفضلة", "سجل الترشيحات", "لوحة المعلومات", "عن المشروع والبيانات"],
        label_visibility="collapsed",
    )
    llm_state = "متصل ✓" if engine.llm_ready else "غير متصل (بحث محلي)"
    db_state = "متصلة ✓" if db else "غير متاحة"
    st.markdown(
        f'<div class="status-box"><b>حالة النظام</b><br>'
        f"النموذج: OpenAI {llm_state}<br>"
        f"قاعدة المتجهات: {len(df)} مطعم<br>"
        f"SQLite: {db_state}</div>",
        unsafe_allow_html=True,
    )
    if st.button("محادثة جديدة"):
        st.session_state.messages = []
        if db:
            try:
                db.clear_messages(SESSION)
            except AppError:
                pass
        st.rerun()
    st.markdown(
        '<div class="team"><b>فريق العمل</b><br>' + "<br>".join(TEAM) + "</div>", unsafe_allow_html=True
    )


# =========================================================================== دوال العرض
def render_result(result: dict, msg_index: int) -> None:
    """يعرض رد المساعد: التفضيلات المفهومة، الملخص، بطاقات المطاعم، والتقييم."""
    badge = (
        '<span class="src-badge">LangChain · RAG · Chroma</span>'
        if result["source"] == "llm"
        else '<span class="src-badge local">بحث محلي احتياطي</span>'
    )
    st.markdown(badge, unsafe_allow_html=True)
    labels = preferences_to_labels(result.get("preferences", {}))
    if labels:
        st.markdown(chips_html(labels, label="فهمت طلبك:"), unsafe_allow_html=True)
    if result.get("warning"):
        st.warning(result["warning"])
    st.markdown(f"**{result['summary']}**")

    recs = result.get("recommendations", [])
    if recs:
        cols = st.columns(len(recs))
        for i, (col, rec) in enumerate(zip(cols, recs), start=1):
            with col:
                info = restaurants_by_id.get(int(rec["restaurant_id"]))
                st.markdown(restaurant_card_html(i, rec, info), unsafe_allow_html=True)
                if st.button("♡ أضف للمفضلة", key=f"fav-{msg_index}-{rec['restaurant_id']}"):
                    try:
                        db.add_favorite(SESSION, rec["restaurant_id"], rec["name"])
                        st.toast(f"تمت إضافة {rec['name']} للمفضلة")
                    except (AppError, AttributeError):
                        st.toast("تعذر الحفظ: قاعدة البيانات غير متاحة")
    if result.get("follow_up_question"):
        st.caption(f"💬 {result['follow_up_question']}")

    c1, c2, _ = st.columns([1, 1, 4])
    for col, (label, helpful) in zip([c1, c2], [("👍 مفيدة", True), ("👎 غير مفيدة", False)]):
        with col:
            if st.button(label, key=f"fb-{msg_index}-{helpful}"):
                try:
                    db.add_feedback(SESSION, result.get("question", ""), helpful)
                    st.toast("شكرًا لتقييمك!")
                except (AppError, AttributeError):
                    st.toast("تعذر حفظ التقييم")


def handle_question(question: str) -> None:
    """يرسل السؤال للـ Engine ويحفظ النتيجة في الجلسة."""
    st.session_state.messages.append({"role": "user", "content": question})
    try:
        with st.spinner("أبحث لك عن أنسب المطاعم…"):
            res = engine.recommend(question, SESSION)
        result = dataclasses.asdict(res)
        result["question"] = question
        st.session_state.messages.append({"role": "assistant", "content": res.summary, "result": result})
    except AppError as exc:  # إدخال خاطئ مثلًا
        st.session_state.messages.append({"role": "assistant", "content": exc.user_message, "error": True})


# =========================================================================== الصفحات
if page == "المحادثة":
    st.markdown(hero_html(settings.chat_model, engine.llm_ready), unsafe_allow_html=True)
    if not engine.llm_ready and engine.init_error:
        st.info(engine.init_error.user_message)

    st.markdown('<span class="chip-label">جرّب:</span>', unsafe_allow_html=True)
    s_cols = st.columns(len(SUGGESTIONS))
    for col, s in zip(s_cols, SUGGESTIONS):
        with col:
            if st.button(s, key=f"sug-{s}"):
                st.session_state.pending = s

    for idx, msg in enumerate(st.session_state.messages):
        with st.chat_message(msg["role"], avatar="🧑" if msg["role"] == "user" else "🍽️"):
            if msg.get("result"):
                render_result(msg["result"], idx)
            elif msg.get("error"):
                st.error(msg["content"])
            else:
                st.markdown(msg["content"])

    typed = st.chat_input("اكتب طلبك… مثال: مطعم ياباني في الرياض للمناسبات")
    question = typed or st.session_state.pending
    if question:
        st.session_state.pending = None
        handle_question(question)
        st.rerun()

elif page == "المفضلة":
    st.header("المطاعم المفضلة")
    favs = db.get_favorites(SESSION) if db else []
    if not favs:
        st.info("ما عندك مطاعم مفضلة للحين. اضغط ♡ على أي ترشيح لحفظه.")
    for f in favs:
        info = restaurants_by_id.get(int(f["restaurant_id"]), {})
        c1, c2 = st.columns([5, 1])
        with c1:
            st.markdown(
                f"**{f['restaurant_name']}** — {info.get('cuisine', '')} · {info.get('city', '')} "
                f"· {PRICE_LABELS.get(int(info.get('price_level', 2)), '')} · {info.get('rating', '')}/5"
            )
        with c2:
            if st.button("حذف", key=f"del-{f['restaurant_id']}"):
                db.remove_favorite(SESSION, f["restaurant_id"])
                st.rerun()

elif page == "سجل الترشيحات":
    st.header("سجل الترشيحات")
    hist = db.get_history(SESSION) if db else []
    if not hist:
        st.info("لا يوجد سجل بعد. ابدأ محادثة من صفحة المحادثة.")
    else:
        table = pd.DataFrame(hist).rename(columns={
            "query": "الطلب", "restaurant_name": "المطعم", "reason": "السبب",
            "match_score": "التطابق %", "source": "المصدر", "created_at": "الوقت",
        })
        st.dataframe(table, use_container_width=True, hide_index=True)

elif page == "لوحة المعلومات":
    st.header("لوحة المعلومات")
    stats = db.get_stats() if db else {}
    c1, c2, c3, c4 = st.columns(4)
    c1.markdown(metric_html(len(df), "مطعم في قاعدة البيانات"), unsafe_allow_html=True)
    c2.markdown(metric_html(stats.get("total_questions", 0), "طلب من المستخدمين"), unsafe_allow_html=True)
    c3.markdown(metric_html(stats.get("total_recommendations", 0), "ترشيح محفوظ"), unsafe_allow_html=True)
    ratio = stats.get("helpful_ratio")
    c4.markdown(metric_html(f"{ratio}%" if ratio is not None else "—", "نسبة الرضا"), unsafe_allow_html=True)
    st.subheader("المطاعم حسب المدينة")
    st.bar_chart(df["city"].value_counts())
    st.subheader("المطاعم حسب نوع المطبخ")
    st.bar_chart(df["cuisine"].value_counts())
    if stats.get("top_restaurants"):
        st.subheader("الأكثر ترشيحًا")
        st.dataframe(pd.DataFrame(stats["top_restaurants"]).rename(
            columns={"restaurant_name": "المطعم", "n": "عدد مرات الترشيح"}), hide_index=True)

else:  # عن المشروع والبيانات
    st.header("عن المشروع")
    st.markdown(
        """
**مُرشِد** نظام يرشّح المطاعم حسب طلب المستخدم المكتوب بلغته العادية.

**مسار الطلب داخل النظام (LangChain Pipeline):**
1. التحقق من المدخلات (Error Handling).
2. **Extraction Chain**: ‏`Prompt | ChatOpenAI.with_structured_output(UserPreferences)` لاستخراج المدينة والمطبخ والسعر.
3. **Retrieval**: البحث الدلالي في **Chroma** باستخدام **OpenAI Embeddings** مع فلترة بالـ metadata.
4. **Recommendation Chain**: ‏`Prompt + Memory | ChatOpenAI.with_structured_output(RecommendationResponse)`.
5. **Guardrail**: نتأكد أن كل مطعم مرشّح موجود فعلًا في نتائج البحث.
6. الحفظ في **SQLite**: الرسائل والترشيحات والمفضلة والتقييم.
"""
    )
    st.subheader("بيانات المطاعم")
    st.caption("بيانات تجريبية أعدّها الفريق لأغراض المشروع (36 مطعمًا بأسماء افتراضية).")
    st.dataframe(df, use_container_width=True, hide_index=True)

    st.subheader("رفع ملف مطاعم جديد (CSV)")
    up = st.file_uploader("ارفع ملف CSV بنفس الأعمدة", type=None)
    if up is not None:
        try:
            if not up.name.lower().endswith(".csv"):
                raise UnsupportedFileError(up.name)
            target = BASE_DIR / "data" / "uploaded_restaurants.csv"
            target.write_bytes(up.getvalue())
            new_df = load_restaurants(target)  # يتحقق من الأعمدة
            st.success(f"الملف صالح ✓ ويحتوي على {len(new_df)} مطعم.")
            st.dataframe(new_df.head(), hide_index=True)
            if st.button("استخدام هذا الملف في الترشيح"):
                st.session_state.data_path = str(target)
                st.rerun()
        except UnsupportedFileError as exc:
            st.error(f"{exc.user_message}\n\nالتفاصيل: {exc.detail}")
    if st.session_state.data_path != str(settings.data_path):
        if st.button("الرجوع لملف المطاعم الأصلي"):
            st.session_state.data_path = str(settings.data_path)
            st.rerun()
''')

### 5.1 معاينة بطاقات المطاعم داخل الدفتر

In [ ]:
from IPython.display import HTML, display
from src.ui.styles import GLOBAL_CSS
from src.ui.components import hero_html, chips_html, restaurant_card_html

info = {int(r["id"]): r for r in df.to_dict("records")}
sample = [
    {"restaurant_id": 9, "name": "لؤلؤة البحر", "match_score": 92, "reason": "مأكولات بحرية طازجة وجلسات عائلية على الكورنيش.", "suggested_dishes": ["روبيان جامبو", "سمك حريد مشوي"]},
    {"restaurant_id": 8, "name": "مذاق الهند", "match_score": 84, "reason": "أرخص ومناسب للعائلات، وفيه أطباق كاري بحري.", "suggested_dishes": ["برياني دجاج", "نان بالثوم"]},
    {"restaurant_id": 5, "name": "بستان الشام", "match_score": 71, "reason": "بديل عائلي بسعر متوسط وحديقة خارجية.", "suggested_dishes": ["مشاوي مشكلة", "فتوش"]},
]
cards = "".join(f'<div style="flex:1">{restaurant_card_html(i, r, info[r["restaurant_id"]])}</div>' for i, r in enumerate(sample, 1))
display(HTML(GLOBAL_CSS + '<link href="https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&display=swap" rel="stylesheet">'
             + '<div dir="rtl" style="font-family:Tajawal;background:#F7F4EE;padding:18px;border-radius:16px">'
             + hero_html("gpt-4o-mini", True) + chips_html(["جدة", "مأكولات بحرية", "مناسب للعائلة"], label="فهمت طلبك:")
             + f'<div style="display:flex;gap:14px">{cards}</div></div>'))
print("* معاينة لشكل البطاقات بقيم مثال")

---
# 6) القسم المشترك — الاختبارات الآلية وتشغيل النظام الكامل

In [ ]:
#@title tests/test_core.py (الكود مخفي) { display-mode: "form" }
from pathlib import Path

def _w(path, text):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8'); print('✓', path)

_w('tests/test_core.py', r'''"""
اختبارات وحدات المشروع (المسؤولة: لمى الأسمري)
التشغيل: pytest -q
هذه الاختبارات لا تحتاج مفتاح OpenAI ولا إنترنت.
"""
import sys
from pathlib import Path

import pandas as pd
import pytest

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

from src.chain import validate_user_input  # noqa: E402
from src.data_loader import load_restaurants, row_to_text, validate_dataframe  # noqa: E402
from src.database import Database  # noqa: E402
from src.errors import InvalidUserInputError, UnsupportedFileError  # noqa: E402
from src.fallback import extract_preferences_rule_based, local_search  # noqa: E402

DATA = ROOT / "data" / "restaurants.csv"


@pytest.fixture
def df():
    return load_restaurants(DATA)


@pytest.fixture
def db(tmp_path):
    return Database(tmp_path / "test.db")


# ------------------------------------------------------------------ البيانات
def test_dataset_loads_with_enough_restaurants(df):
    assert len(df) >= 30
    assert df["id"].is_unique


def test_text_contains_key_fields(df):
    text = row_to_text(df.iloc[0])
    assert "اسم المطعم" in text and "التقييم" in text and df.iloc[0]["name"] in text


def test_unsupported_extension(tmp_path):
    f = tmp_path / "data.txt"
    f.write_text("x")
    with pytest.raises(UnsupportedFileError):
        load_restaurants(f)


def test_missing_columns():
    with pytest.raises(UnsupportedFileError):
        validate_dataframe(pd.DataFrame({"name": ["a"]}))


# ------------------------------------------------------------------ المدخلات
@pytest.mark.parametrize("bad", ["", "  ", "؟؟", "؟؟؟؟؟", "!!!!!", "123456", "a" * 501])
def test_invalid_inputs_rejected(bad):
    with pytest.raises(InvalidUserInputError):
        validate_user_input(bad)


def test_valid_input_passes():
    assert validate_user_input("  مطعم في جدة ") == "مطعم في جدة"


# ------------------------------------------------------------------ البحث الاحتياطي
def test_rule_based_extraction():
    p = extract_preferences_rule_based("أبي مطعم بحري رخيص في جده للعائلة")
    assert p["city"] == "جدة"
    assert p["cuisine"] == "مأكولات بحرية"
    assert p["max_price_level"] == 1
    assert p["family_friendly"] is True


def test_local_search_respects_city(df):
    recs = local_search(df, "مطعم بحري في جدة")
    assert recs, "يجب أن يرجع نتائج"
    ids = [r["restaurant_id"] for r in recs]
    cities = df.set_index("id").loc[ids, "city"].tolist()
    assert cities[0] == "جدة"
    assert all(0 <= r["match_score"] <= 100 for r in recs)


# ------------------------------------------------------------------ قاعدة البيانات
def test_messages_and_window(db):
    sid = db.create_session()
    for i in range(10):
        db.add_message(sid, "user" if i % 2 == 0 else "assistant", f"رسالة {i}")
    last = db.get_messages(sid, limit=4)
    assert [m["content"] for m in last] == ["رسالة 6", "رسالة 7", "رسالة 8", "رسالة 9"]


def test_favorites_no_duplicates(db):
    sid = db.create_session()
    db.add_favorite(sid, 9, "لؤلؤة البحر")
    db.add_favorite(sid, 9, "لؤلؤة البحر")
    assert len(db.get_favorites(sid)) == 1
    db.remove_favorite(sid, 9)
    assert db.get_favorites(sid) == []


def test_recommendations_and_stats(db):
    sid = db.create_session()
    db.add_message(sid, "user", "مطعم بحري")
    db.save_recommendations(sid, "مطعم بحري", [
        {"restaurant_id": 9, "name": "لؤلؤة البحر", "reason": "بحري", "match_score": 90},
    ])
    db.add_feedback(sid, "مطعم بحري", True)
    stats = db.get_stats()
    assert stats["total_questions"] == 1
    assert stats["total_recommendations"] == 1
    assert stats["helpful_ratio"] == 100
    assert db.get_history(sid)[0]["restaurant_name"] == "لؤلؤة البحر"


# ------------------------------------------------------------------ المحرك بدون مفتاح (وضع الطوارئ)
def test_engine_without_api_key_uses_local_search(df, db, tmp_path):
    import dataclasses

    from src.config import get_settings
    from src.chain import RecommendationEngine

    settings = dataclasses.replace(get_settings(), openai_api_key="", db_path=tmp_path / "x.db")
    engine = RecommendationEngine(settings, df, db)
    assert not engine.llm_ready
    sid = db.create_session()
    res = engine.recommend("مطعم بحري عائلي في جدة", sid)
    assert res.source == "local"
    assert res.warning  # رسالة توضح أن المفتاح غير موجود
    assert 1 <= len(res.recommendations) <= 3
    # تم حفظ السؤال والرد والترشيحات
    assert len(db.get_messages(sid)) == 2
    assert len(db.get_history(sid)) == len(res.recommendations)


def test_local_follow_up_keeps_city(df, db, tmp_path):
    import dataclasses

    from src.config import get_settings
    from src.chain import RecommendationEngine

    settings = dataclasses.replace(get_settings(), openai_api_key="", db_path=tmp_path / "y.db")
    engine = RecommendationEngine(settings, df, db)
    sid = db.create_session()
    first = engine.recommend("مطعم بحري عائلي في جدة", sid)
    second = engine.recommend("طيب أبي أرخص", sid)
    assert second.preferences.get("city") == "جدة"
    cities = df.set_index("id").loc[[r["restaurant_id"] for r in second.recommendations], "city"]
    assert cities.iloc[0] == "جدة"
    assert first.recommendations
''')

### 6.1 تشغيل الاختبارات الآلية (18 اختبار)

In [ ]:
!OPENAI_API_KEY= python -m pytest -q tests/

### 6.2 تشغيل الواجهة (الديمو الحي)
خلية وحدة: تشغّل التطبيق وتعطيك زر يفتح الواجهة. خليها شغالة طول ما تستخدمين التطبيق.

In [ ]:
import os, re, subprocess, threading, time, urllib.request
from pathlib import Path
from IPython.display import HTML, display

needed = ["app.py", "src/chain.py", "src/database.py", "src/vector_store.py", "src/ui/components.py", "data/restaurants.csv"]
missing = [f for f in needed if not Path(f).exists()]
if missing:
    raise RuntimeError("ملفات ناقصة: " + "، ".join(missing) + " — شغلي كل الخلايا من فوق (Runtime ← Run all)")

subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
subprocess.Popen(["streamlit", "run", "app.py"], stdout=open("/content/streamlit_logs.txt", "w"), stderr=subprocess.STDOUT)

for _ in range(90):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=1); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("التطبيق ما اشتغل — شوفي سجل التشغيل في الخلية اللي تحت")

url = None
try:
    if not os.path.exists("cloudflared"):
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
        os.chmod("cloudflared", 0o755)
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t0 = time.time()
    for line in proc.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0); break
        if time.time() - t0 > 60:
            break
    if url:
        threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
        time.sleep(5)
except Exception as error:
    print("رابط cloudflare ما اشتغل:", error)

if not url:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(8501)")

display(HTML(
    f'<a href="{url}" target="_blank" style="display:inline-block;padding:12px 28px;background:#12808A;'
    'border-radius:14px;font:700 16px Tajawal,sans-serif;color:#fff;text-decoration:none">🍽️ افتحي واجهة مُرشِد</a>'
    f'<p dir="ltr" style="color:#6B7780">{url}</p>'))

### 6.3 (اختياري) سجل التشغيل إذا صارت مشكلة

In [ ]:
!tail -n 30 /content/streamlit_logs.txt

### 6.4 تجميع ملفات المشروع للتسليم (Source Code + قاعدة البيانات)

In [ ]:
!zip -q -r murshid_project.zip app.py src data tests assets static .streamlit requirements.txt .env.example .gitignore restaurant_app.db demo.html -x "*/__pycache__/*"
print("تم تجميع المشروع ✓ (ملف .env ما انضاف لأن فيه المفتاح)")
from google.colab import files
files.download("murshid_project.zip")